# P04 · Notebook 26 · The second view on MIMII's noise levels (valves and pumps at 6 dB)

**Where we stand** (notebook 23: the frozen MIC system, applied unchanged at each level; each cell is Split-A AUC / arXiv 2309.07500; bold is below the published number):

| Level | Pump | Fan | Valve | Slide rail |
|---|---|---|---|---|
| 6 dB | **0.9875** / 0.9891 | 0.9968 / 0.9682 | **0.9951** / 0.9978 | 0.9990 / 0.9974 |
| 0 dB | 0.9679 / 0.9115 | 0.9675 / 0.9260 | 0.9902 / 0.9830 | 0.9929 / 0.9750 |
| −6 dB | 0.9485 / 0.8977 | 0.8911 / 0.8778 | 0.9830 / 0.9794 | 0.9706 / 0.9441 |

**What it does:**
1. **Checks first (minutes, before any download).** The frozen system's level tables come back from the embeddings saved in Drive. No BEATs or ECHO model is loaded.
2. **Trains notebook 25's view again, three seeds,** on the program's own split of MIMII's microphone 0 at −6 dB (28 classes, as in notebook 25).
3. **Decides at −6 dB.** The frozen system fused with the view (form ZP, the form notebook 25 selected) is kept only if pumps and fans rise by 0.01 on Split A and valves and slide rails lose no more than 0.01.
4. **Only if it is kept:** the same system is read once at 6 and 0 dB against the published numbers.

**Rules:** `results/mimii_beats/RULES_second_view_mimii_written_before_running.md`, written before this notebook read any real data.

**It needs** in Google Drive: `P04_DenoiseScore` (notebook 20), `P04_StrongFineTune` (notebook 21), `P04_NewEncoders` (notebook 22b) and `P04_ExternalJudgement` (notebook 23). It writes to a new folder, `P04_SecondViewMIMII` (1 GB is enough).

**Runtime:**
- Choose **G4** (RTX PRO 6000): Runtime › Change runtime type. Then Runtime › Run all.
- About 100 GB of downloads in all (MIMII at three levels; only microphone 0 is kept), and three trainings of 300 epochs.
- **Time:** about 2–3 hours on G4. Notebook 25 measured this model there at 0.019–0.020 s per training step. A T4 was never measured for this model; it could take 8–20 hours. Cell 7 measures this GPU in the first minute of training and prints its own estimate.
- Everything needed is saved to Drive. If Colab disconnects, run all again: finished trainings are skipped, an unfinished one continues from its last save (every 10 epochs), and audio is downloaded again only where it is still needed.

**At the end** `p04_n26_results.zip` downloads. Send it back here.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, math, gc, threading
from pathlib import Path
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from sklearn.metrics import roc_auc_score
try:
    import torchaudio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
    import torchaudio

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK = MY / "P04_SecondViewMIMII", Path("/content/work")
    N20, N21, N22B, N23 = MY / "P04_DenoiseScore", MY / "P04_StrongFineTune", MY / "P04_NewEncoders", MY / "P04_ExternalJudgement"
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_SecondViewMIMII")), Path(os.environ.get("P04_WORK", "work"))
    N20, N21 = Path(os.environ.get("P04_N20_DIR", "P04_DenoiseScore")), Path(os.environ.get("P04_N21_DIR", "P04_StrongFineTune"))
    N22B, N23 = Path(os.environ.get("P04_N22B_DIR", "P04_NewEncoders")), Path(os.environ.get("P04_N23_DIR", "P04_ExternalJudgement"))
for p, who in ((N20, "notebook 20 (P04_DenoiseScore)"), (N21, "notebook 21 (P04_StrongFineTune)"), (N22B, "notebook 22b (P04_NewEncoders)"),
               (N23, "notebook 23 (P04_ExternalJudgement)")):
    if not p.exists():
        raise SystemExit(f"missing {p}: the folder of {who} must be in Google Drive › MyDrive")
for p in (OUT, WORK):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
if DEVICE != "cuda" and not TEST:
    raise SystemExit("No GPU. Runtime › Change runtime type › G4 (or another GPU), then Runtime › Run all.")
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True                             # fixed input sizes: cuDNN picks its fastest algorithms
T_START = time.time()
DRIVE_FREE_GB = shutil.disk_usage(OUT).free / 1e9
print("device:", DEVICE, GPU, f"({GPU_GB:.0f} GB)" if GPU else "", "· torch", torch.__version__, "· torchaudio", torchaudio.__version__,
      f"· free local disk {shutil.disk_usage(WORK).free / 1e9:.0f} GB · free Drive {DRIVE_FREE_GB:.1f} GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · Every number below was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_second_view_mimii_written_before_running.md"
# MIMII on Zenodo (12 zips: four types at three levels) and DCASE 2020's additional training data (notebook 23's lines)
Z = "https://zenodo.org/records/3384388/files/"
MD5 = {"pump|6dB": "a09ba6060c10fc09cd4c8770213b0b9f", "pump|-6dB": "d20b783a0ff9c93d58f452f98c37b112",
       "fan|6dB": "0890f7d3c2fd8448634e69ff1d66dd47", "fan|-6dB": "f02ae808a58d84b6815b7ec38ff30879",
       "valve|6dB": "fe5fb7c337cd701b1d31dc641e621892", "valve|-6dB": "fdfaf185fea61b21e11952a070a4ada7",
       "slider|6dB": "838c2b3441858359c4704ef13a1b27ff", "slider|-6dB": "49913eda7d37f182cbf8ed5c984140e0"}
ZIPS = {}
for key, h in MD5.items():
    t_, lv_ = key.split("|"); nm = f"{lv_.replace('dB', '_dB')}_{t_}.zip"
    ZIPS[key] = (Z + nm + "?download=1", nm, h)
MD5_0 = {"pump": "488748295c3f60b25de07b58fe75b049", "fan": "6354d1cc2165c52168f9ef1bcd9c7c52",
         "valve": "178478eb0d11c79080a35562bfdeee71", "slider": "4d674c21474f0646ecd75546db6c0c4e"}      # MIMII at 0 dB (notebook 19)
ZIPS0 = {t_: (Z + f"0_dB_{t_}.zip?download=1", f"0_dB_{t_}.zip", h) for t_, h in MD5_0.items()}
ZX = "https://zenodo.org/records/3727685/files/"                    # DCASE 2020 Task 2, additional training dataset
XMD5 = {"pump": "4e33ae9c0db5cc88437675f5e317caee", "fan": "567798854130c8019df8b664c095ed1e",
        "valve": "057cdd560caffb8305449541afd28c6d", "slider": "f84f186f4295fc23754abcdd1b0580a7"}
XZIPS = {t_: (ZX + f"eval_data_train_{t_}.zip?download=1", f"eval_data_train_{t_}.zip", h) for t_, h in XMD5.items()}
ZIPS.update({f"{t_}|0dB": v for t_, v in ZIPS0.items()})              # one table for the three levels: "type|level" → (url, name, MD5)
GBZ = {"pump": 7.9, "fan": 10.4, "valve": 7.4, "slider": 7.5}         # approximate size of one level's zip of a type, GB (notebook 19)
FREE_MARGIN_GB = 8.0                                                  # a zip downloads in the background only if the disk keeps this much free
TYPES, MAIN, AUX = ["pump", "fan", "valve", "slider"], ["pump", "fan"], ["valve", "slider"]
LVL = ["6dB", "0dB", "-6dB"]; LEVEL_CODE = {"-6dB": "m6", "0dB": "0", "6dB": "6"}
LV_NAME = {"6dB": "6 dB", "0dB": "0 dB", "-6dB": "−6 dB"}
DECISION_LEVEL, OTHER_LEVELS = "-6dB", ["6dB", "0dB"]                 # the program decides at −6 dB; the other levels are read only if kept
SR, LDN_FLOOR, BAR = 16000, 1e-6, 0.01
SHOTS = [1, 3, 10]
SPLITS = ["A", "B", "B+1", "B+3", "B+10"]
FOLDS = {1: ["id_00", "id_02"], 2: ["id_04", "id_06"]}
# the second view: notebook 25's settings, unchanged (github.com/soonhyeon/Noisy-ArcMix at REF_COMMIT)
REF_COMMIT = "674c759bfaa7723619a4b1d8d00eb9d155c50ba6"
SV = dict(n_fft=2048, hop=512, n_mels=128, win_len=1024, c_dim=128, arc_s=30.0, arc_m=0.7, alpha=0.5, lr=1e-4, eta_min_ratio=0.1,
          epochs=300, batch=64, split_ratio=0.95, seeds=[0, 1, 2], sine_floor=1e-9, save_every=10, embed_batch=64,
          gpu_headroom_gb=5.0)
REF_FRAMES, REF_KERNEL, REF_CLASSES, REF_CLIPS, REF_TRAIN, REF_MIMII, REF_XC = 313, (8, 20), 28, 21583, 20503, 11419, 10164
CAL_MIN = 30                                                          # a spread needs at least 30 clips per type
# arXiv 2309.07500, Table 1 (one microphone): the published Split-A numbers the per-level table is read against
PUBLISHED = {'6dB': {'pump': 0.9891, 'fan': 0.9682, 'valve': 0.9978, 'slider': 0.9974}, '0dB': {'pump': 0.9115, 'fan': 0.926, 'valve': 0.983, 'slider': 0.975}, '-6dB': {'pump': 0.8977, 'fan': 0.8778, 'valve': 0.9794, 'slider': 0.9441}}
TARGETS = [("6dB", "pump"), ("6dB", "valve")]                         # the two cells reading 1 missed
# notebook 23's MIC tables (reading 1) at each level: every split, machine and average, from its results zip
# (MD5 8301df3f5c591db368a94265fd352cef). Gate G0(c) compares this session's recomputation with them.
N23_LEVELS = {'6dB': {'pump': {'A': {'id_00': 0.9969, 'id_02': 0.9744, 'id_04': 0.9831, 'id_06': 0.9955, 'avg': 0.9875}, 'B': {'id_00': 0.902, 'id_02': 0.9502, 'id_04': 0.1931, 'id_06': 0.691, 'avg': 0.6841}, 'B+1': {'id_00': 0.9993, 'id_02': 1.0, 'id_04': 0.9816, 'id_06': 0.9996, 'avg': 0.9951}, 'B+3': {'id_00': 0.9904, 'id_02': 1.0, 'id_04': 0.8946, 'id_06': 0.9962, 'avg': 0.9703}, 'B+10': {'id_00': 0.9991, 'id_02': 1.0, 'id_04': 0.8873, 'id_06': 0.9979, 'avg': 0.9711}}, 'fan': {'A': {'id_00': 0.991, 'id_02': 1.0, 'id_04': 0.9986, 'id_06': 0.9974, 'avg': 0.9968}, 'B': {'id_00': 0.3923, 'id_02': 0.2504, 'id_04': 0.6719, 'id_06': 0.2658, 'avg': 0.3951}, 'B+1': {'id_00': 0.7989, 'id_02': 0.984, 'id_04': 0.9831, 'id_06': 0.9582, 'avg': 0.9311}, 'B+3': {'id_00': 0.806, 'id_02': 0.9776, 'id_04': 0.9866, 'id_06': 0.9644, 'avg': 0.9336}, 'B+10': {'id_00': 0.8497, 'id_02': 0.9863, 'id_04': 0.9839, 'id_06': 0.9719, 'avg': 0.9479}}, 'valve': {'A': {'id_00': 1.0, 'id_02': 0.9999, 'id_04': 0.999, 'id_06': 0.9816, 'avg': 0.9951}, 'B': {'id_00': 0.9984, 'id_02': 0.1893, 'id_04': 0.4004, 'id_06': 0.3194, 'avg': 0.4769}, 'B+1': {'id_00': 1.0, 'id_02': 0.7618, 'id_04': 0.9904, 'id_06': 0.8288, 'avg': 0.8953}, 'B+3': {'id_00': 1.0, 'id_02': 0.8804, 'id_04': 0.9898, 'id_06': 0.6986, 'avg': 0.8922}, 'B+10': {'id_00': 1.0, 'id_02': 0.8581, 'id_04': 0.9876, 'id_06': 0.9603, 'avg': 0.9515}}, 'slider': {'A': {'id_00': 1.0, 'id_02': 0.9994, 'id_04': 0.9971, 'id_06': 0.9997, 'avg': 0.999}, 'B': {'id_00': 0.9965, 'id_02': 1.0, 'id_04': 0.2585, 'id_06': 0.7854, 'avg': 0.7601}, 'B+1': {'id_00': 1.0, 'id_02': 1.0, 'id_04': 0.8928, 'id_06': 0.7139, 'avg': 0.9017}, 'B+3': {'id_00': 1.0, 'id_02': 1.0, 'id_04': 0.9454, 'id_06': 0.7287, 'avg': 0.9185}, 'B+10': {'id_00': 1.0, 'id_02': 1.0, 'id_04': 0.9801, 'id_06': 0.8854, 'avg': 0.9664}}}, '0dB': {'pump': {'A': {'id_00': 0.9915, 'id_02': 0.9348, 'id_04': 0.9546, 'id_06': 0.9907, 'avg': 0.9679}, 'B': {'id_00': 0.7343, 'id_02': 0.6857, 'id_04': 0.4036, 'id_06': 0.6845, 'avg': 0.627}, 'B+1': {'id_00': 0.9679, 'id_02': 1.0, 'id_04': 0.9531, 'id_06': 0.9995, 'avg': 0.9801}, 'B+3': {'id_00': 0.9735, 'id_02': 0.9994, 'id_04': 0.9266, 'id_06': 0.9967, 'avg': 0.9741}, 'B+10': {'id_00': 0.9779, 'id_02': 0.9998, 'id_04': 0.8552, 'id_06': 0.9879, 'avg': 0.9552}}, 'fan': {'A': {'id_00': 0.9109, 'id_02': 0.981, 'id_04': 0.9841, 'id_06': 0.994, 'avg': 0.9675}, 'B': {'id_00': 0.6467, 'id_02': 0.4563, 'id_04': 0.7044, 'id_06': 0.2465, 'avg': 0.5135}, 'B+1': {'id_00': 0.5372, 'id_02': 0.864, 'id_04': 0.8185, 'id_06': 0.8835, 'avg': 0.7758}, 'B+3': {'id_00': 0.5205, 'id_02': 0.9123, 'id_04': 0.8996, 'id_06': 0.8598, 'avg': 0.798}, 'B+10': {'id_00': 0.6598, 'id_02': 0.8163, 'id_04': 0.9289, 'id_06': 0.9571, 'avg': 0.8405}}, 'valve': {'A': {'id_00': 1.0, 'id_02': 0.9999, 'id_04': 0.9742, 'id_06': 0.9867, 'avg': 0.9902}, 'B': {'id_00': 0.9999, 'id_02': 0.1673, 'id_04': 0.7143, 'id_06': 0.4726, 'avg': 0.5885}, 'B+1': {'id_00': 0.9999, 'id_02': 0.6788, 'id_04': 0.9808, 'id_06': 0.9617, 'avg': 0.9053}, 'B+3': {'id_00': 1.0, 'id_02': 0.3653, 'id_04': 0.972, 'id_06': 0.9424, 'avg': 0.8199}, 'B+10': {'id_00': 1.0, 'id_02': 0.8036, 'id_04': 0.9735, 'id_06': 0.9673, 'avg': 0.9361}}, 'slider': {'A': {'id_00': 0.9999, 'id_02': 0.999, 'id_04': 0.9977, 'id_06': 0.9748, 'avg': 0.9929}, 'B': {'id_00': 0.9999, 'id_02': 0.9993, 'id_04': 0.1501, 'id_06': 0.6055, 'avg': 0.6887}, 'B+1': {'id_00': 1.0, 'id_02': 1.0, 'id_04': 0.8049, 'id_06': 0.7287, 'avg': 0.8834}, 'B+3': {'id_00': 1.0, 'id_02': 1.0, 'id_04': 0.814, 'id_06': 0.6278, 'avg': 0.8605}, 'B+10': {'id_00': 1.0, 'id_02': 1.0, 'id_04': 0.9273, 'id_06': 0.6773, 'avg': 0.9011}}}, '-6dB': {'pump': {'A': {'id_00': 0.9435, 'id_02': 0.9403, 'id_04': 0.9115, 'id_06': 0.9986, 'avg': 0.9485}, 'B': {'id_00': 0.6666, 'id_02': 0.6817, 'id_04': 0.4207, 'id_06': 0.6689, 'avg': 0.6095}, 'B+1': {'id_00': 0.7276, 'id_02': 0.7617, 'id_04': 0.9867, 'id_06': 0.7823, 'avg': 0.8146}, 'B+3': {'id_00': 0.7884, 'id_02': 0.7706, 'id_04': 0.9094, 'id_06': 0.8159, 'avg': 0.8211}, 'B+10': {'id_00': 0.9572, 'id_02': 0.9457, 'id_04': 0.7929, 'id_06': 0.9316, 'avg': 0.9069}}, 'fan': {'A': {'id_00': 0.6857, 'id_02': 0.9627, 'id_04': 0.9178, 'id_06': 0.9982, 'avg': 0.8911}, 'B': {'id_00': 0.5169, 'id_02': 0.3963, 'id_04': 0.512, 'id_06': 0.3007, 'avg': 0.4315}, 'B+1': {'id_00': 0.5108, 'id_02': 0.4052, 'id_04': 0.5129, 'id_06': 0.8573, 'avg': 0.5716}, 'B+3': {'id_00': 0.5407, 'id_02': 0.596, 'id_04': 0.5725, 'id_06': 0.6047, 'avg': 0.5785}, 'B+10': {'id_00': 0.5489, 'id_02': 0.7578, 'id_04': 0.5994, 'id_06': 0.8074, 'avg': 0.6784}}, 'valve': {'A': {'id_00': 1.0, 'id_02': 1.0, 'id_04': 0.9528, 'id_06': 0.9792, 'avg': 0.983}, 'B': {'id_00': 0.9224, 'id_02': 0.1397, 'id_04': 0.3369, 'id_06': 0.4958, 'avg': 0.4737}, 'B+1': {'id_00': 0.9819, 'id_02': 0.215, 'id_04': 0.9385, 'id_06': 0.9182, 'avg': 0.7634}, 'B+3': {'id_00': 0.984, 'id_02': 0.5405, 'id_04': 0.9528, 'id_06': 0.9201, 'avg': 0.8494}, 'B+10': {'id_00': 0.9937, 'id_02': 0.6622, 'id_04': 0.9578, 'id_06': 0.9762, 'avg': 0.8975}}, 'slider': {'A': {'id_00': 0.999, 'id_02': 0.971, 'id_04': 0.9809, 'id_06': 0.9316, 'avg': 0.9706}, 'B': {'id_00': 0.9301, 'id_02': 0.9347, 'id_04': 0.313, 'id_06': 0.5931, 'avg': 0.6927}, 'B+1': {'id_00': 0.9572, 'id_02': 0.9787, 'id_04': 0.8104, 'id_06': 0.7192, 'avg': 0.8664}, 'B+3': {'id_00': 0.994, 'id_02': 0.9845, 'id_04': 0.8179, 'id_06': 0.7515, 'avg': 0.887}, 'B+10': {'id_00': 0.993, 'id_02': 0.988, 'id_04': 0.8892, 'id_06': 0.8053, 'avg': 0.9189}}}}
CURRENT_A, KEEP_AT = (0.9198, 0.9768), (0.9298, 0.9668)                       # Split A at −6 dB (development, confirmation); the keep rule's two bars
G0C_TOL, DRIVE_MIN_GB, DISK_MIN_GB = 0.0005, 1.0, 45.0
DCASE_BASELINE_DEV = {"fan": 0.6583, "pump": 0.7289, "slider": 0.8476, "valve": 0.6628}   # the task page, mean of 10 trials (gate G-VIEW here)
# notebook 25, for comparison only: its spreads on DCASE's lists, and the view alone on DCASE's development set (the paper's score)
N25_SIGMA = {'frozen': {'pump': 0.2035, 'fan': 0.1925, 'valve': 0.1696, 'slider': 0.2167}, 'view_paper': {'pump': 0.3741, 'fan': 0.249, 'valve': 0.5964, 'slider': 0.3747}}
N25_VIEW_DEV = {'pump': 0.9616, 'fan': 0.9793, 'valve': 0.9983, 'slider': 0.9961}
FORCE = None
if TEST:
    _d = json.loads(os.environ["P04_TEST_DATA"])
    XZIPS = {t_: tuple(v) for t_, v in _d.pop("xc").items()}
    ZIPS = {k: tuple(v) for k, v in _d.items()}
    N23_LEVELS = {lv: json.loads((N23 / "external_judgement.json").read_text())["levels"][f"MIC|{lv}"] for lv in LVL}   # the stand-in's own tables
    _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    SV.update(_c.get("sv", {})); CAL_MIN = _c.get("cal_min", CAL_MIN)
    GBZ = {t_: 0.0 for t_ in TYPES}; FREE_MARGIN_GB = _c.get("free_margin_gb", 0.0)   # the test's zips are small
    FORCE = os.environ.get("P04_TEST_FORCE")                          # tests only: walk the kept / not-kept branch
assert sorted(ZIPS) == sorted(f"{t_}|{lv_}" for t_ in TYPES for lv_ in LVL), "the zip table does not hold four types at three levels"
print("rules:", RULES, "·", json.dumps(SV))
if not TEST and DRIVE_FREE_GB < DRIVE_MIN_GB:
    raise SystemExit(f"stop: Drive has {DRIVE_FREE_GB:.1f} GB free and this run needs {DRIVE_MIN_GB} GB. Free some space in Google Drive, then run again.")

In [ ]:
# 3 · Notebook 20's clip lists and the program's splits; the frozen MIC system from its saved embeddings; gates G0(a) and G0(c).
# No BEATs or ECHO model is loaded and nothing is downloaded yet: a missing or wrong file stops the run here, within minutes.
def unit(E):
    return E / np.linalg.norm(E, axis=-1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def ldn1(R, X):
    # Wilkinghoff et al. (TASLP 2026), ratio form with K = 1: min over references y of d(x, y) / d(y, y₁)
    U = unit(R); Srr = U @ U.T; np.fill_diagonal(Srr, -np.inf)
    d1 = np.maximum(1.0 - Srr.max(1), LDN_FLOOR)
    return ((1.0 - unit(X) @ U.T) / d1[None, :]).min(1)


def split_a(pump, label, fname, p):
    # the same split as notebooks 1 and 3: test = first N normal by name + all abnormal of pump p; train = the other normal
    idx = np.where(pump == p)[0]
    nor = idx[label[idx] == 0]; abn = idx[label[idx] == 1]
    nor = nor[np.argsort(fname[nor])]
    return nor[len(abn):], np.r_[nor[:len(abn)], abn]


def spread(logs):
    # the centre and the robust spread (1.4826 × the median absolute deviation) of a system's log scores on normal clips
    if len(logs) == 0:
        return {"n": 0, "median": None, "sigma": None}
    med = float(np.median(logs))
    return {"n": int(len(logs)), "median": med, "sigma": float(1.4826 * np.median(np.abs(logs - med)))}


def fold_of(p):
    return 1 if p in FOLDS[1] else 2


def side(p, s):
    # the fold whose models score machine p in split s: its own fold for A, the other fold for B and B+n (notebook 23)
    own = fold_of(p)
    return own if s == "A" else 3 - own


FROZEN_PARTS = [("BEATs", N21, "MIC_XC"), ("BEATs-iter3", N22B, "B3_MIC_XC")]   # encoder, the folder of its −6 dB embeddings, the file tag
FROZEN_SEEDS = [0, 1, 2]


def f_m6(part, s, fold):
    return part[1] / f"emb_{part[2]}_s{s}_f{fold}.npz"


def f_lv(lv, part, s, fold, t):
    return N23 / "levels" / f"{LEVEL_CODE[lv]}__MIC_{part[0]}_{part[2]}_s{s}_f{fold}__{t}.npy"


def f_echo(lv, t):
    return N23 / "levels" / f"{LEVEL_CODE[lv]}__ECHO__{t}.npy"


# G0(a) · every file the frozen system needs is in Drive
_need = [N20 / f"emb_{t}.npz" for t in TYPES] + [f_m6(part, s, k) for part in FROZEN_PARTS for s in FROZEN_SEEDS for k in (1, 2)]
_need += [f_lv(lv, part, s, k, t) for lv in OTHER_LEVELS for part in FROZEN_PARTS for s in FROZEN_SEEDS for k in (1, 2) for t in TYPES]
_need += [f_echo(lv, t) for lv in OTHER_LEVELS for t in TYPES]
_missing = [str(f) for f in _need if not f.exists()]
G0 = {"a": {"ok": not _missing, "files": len(_need), "missing": _missing[:12]}}
print(f"G0(a) · the frozen system's {len(_need)} files are in Drive:", not _missing)
if _missing:
    raise SystemExit(f"stop: G0(a) failed. {len(_missing)} files are missing in Drive, for example {_missing[0]}")

META, ECHO_M6 = {}, {}
for t in TYPES:
    D = np.load(N20 / f"emb_{t}.npz", allow_pickle=False)
    META[t] = (D["fname"], D["mid"], D["label"]); ECHO_M6[t] = D["ECHO__CH0"].astype(np.float64)
SPLIT = {t: {p: split_a(META[t][1], META[t][2], META[t][0], p) for p in sorted(set(META[t][1].tolist()))} for t in TYPES}
TRAIN_ROWS = {(t, p): SPLIT[t][p][0] for t in TYPES for p in SPLIT[t]}
IDS = {t: sorted(SPLIT[t]) for t in TYPES}
print("MIMII clips per type:", {t: len(META[t][0]) for t in TYPES}, "· machines:", IDS[TYPES[0]],
      "· normal training clips per type (the program's split):", {t: int(sum(len(TRAIN_ROWS[(t, p)]) for p in IDS[t])) for t in TYPES})
FM = {}


def frozen_matrix(t, lv, pi, fold):
    # one encoder's matrix for a fold's models at a level (notebook 23's algebra, unchanged): the three seeds' F unit-normalised,
    # concatenated, ÷√3; ECHO concatenated, ÷√2
    k = (t, lv, pi, fold)
    if k not in FM:
        part = FROZEN_PARTS[pi]
        if lv == "-6dB":
            Es = [np.load(f_m6(part, s, fold), allow_pickle=False)[f"F__{t}"].astype(np.float32) for s in FROZEN_SEEDS]
            echo = ECHO_M6[t]
        else:
            Es = [np.asarray(np.load(f_lv(lv, part, s, fold, t), mmap_mode="r")).astype(np.float32) for s in FROZEN_SEEDS]
            echo = np.asarray(np.load(f_echo(lv, t), mmap_mode="r"), np.float64)
        E = np.concatenate([unit(e) for e in Es], 1) / math.sqrt(len(Es))
        FM[k] = np.concatenate([E, unit(echo)], 1) / math.sqrt(2.0)
        assert len(FM[k]) == len(META[t][0]), f"the saved embeddings of {t} at {lv} do not have notebook 20's number of clips"
    return FM[k]


def frozen_score(t, lv, p, s, ref, te):
    # the mean of the two encoders' LDN1 scores, each with the models of the fold that serves machine p in split s
    k = side(p, s); out = 0.0
    for pi in range(len(FROZEN_PARTS)):
        E = frozen_matrix(t, lv, pi, k)
        out = out + (1.0 / len(FROZEN_PARTS)) * ldn1(E[ref], E[te])
    return out


def ldn1_left_out(R, pos):
    # LDN1 of the reference clips at positions `pos`, each against all the other references (the clip itself left out)
    U = unit(R); S = U @ U.T; np.fill_diagonal(S, -np.inf); out = []
    for h in pos:
        keep = np.ones(len(U), bool); keep[h] = False
        d1 = np.maximum(1.0 - S[np.ix_(keep, keep)].max(1), LDN_FLOOR)
        out.append(float(((1.0 - S[h, keep]) / d1).min()))
    return np.array(out)


def ldn1_left_out_all(R):
    # LDN1 of every reference against all the others (the clip itself left out). With clip h left out, a reference y's own
    # nearest neighbour is searched without h: it is y's second neighbour when h was its first.
    U = unit(R); S = U @ U.T; np.fill_diagonal(S, -np.inf)
    top2 = np.partition(S, -2, axis=1)[:, -2:]                        # per reference: its second largest and its largest similarity
    s2_, s1_, a1 = top2[:, 0], top2[:, 1], S.argmax(1)
    out = np.empty(len(U))
    for h in range(len(U)):
        d1 = np.maximum(1.0 - np.where(a1 == h, s2_, s1_), LDN_FLOOR)
        r = (1.0 - S[h]) / d1; r[h] = np.inf
        out[h] = r.min()
    return out


_R = np.random.default_rng(5).standard_normal((30, 12))
assert np.allclose(ldn1_left_out_all(_R), ldn1_left_out(_R, range(30))), "stop: the fast left-out LDN1 is not notebook 25's left-out LDN1"
assert np.allclose(ldn1_left_out_all(_R)[[3, 17]], [ldn1(np.delete(_R, h, 0), _R[[h]])[0] for h in (3, 17)]), "stop: the left-out LDN1 is not LDN1"


def evaluate_fn(fn, t):
    # notebook 11's splits; fn(p, split, reference rows, test rows) → anomaly scores of the test rows
    ids = sorted(SPLIT[t]); label = META[t][2]
    r = {s: {} for s in SPLITS}
    for p in ids:
        tr, te = SPLIT[t][p]; y = label[te]
        fleet = np.concatenate([SPLIT[t][q][0] for q in ids if q != p])
        for s, ref in (("A", tr), ("B", fleet), *[(f"B+{n}", np.r_[fleet, tr[:n]]) for n in SHOTS]):
            r[s][p] = round(float(roc_auc_score(y, fn(p, s, ref, te))), 4)
    for s in SPLITS:
        r[s]["avg"] = round(float(np.mean([r[s][p] for p in ids])), 4)
    return r


# G0(c) · the frozen system comes back at every level: all five splits, every machine and every average of notebook 23's MIC tables.
# On the way, two things are kept for later: the Split-A scores of every test clip (SF), and every normal training clip's score
# against its machine's other training clips (LOO_F), from which the frozen system's spread is measured (the rules).
t_g0 = time.time()
SF, LOO_F, FROZEN_RES, worst, n_cmp = {}, {}, {}, 0.0, 0
for lv in LVL:
    FROZEN_RES[lv] = {}
    for t in TYPES:
        def _fn(p, s, ref, te, lv=lv, t=t):
            sc = frozen_score(t, lv, p, s, ref, te)
            if s == "A":
                SF[(lv, t, p)] = sc
            return sc
        r = evaluate_fn(_fn, t); FROZEN_RES[lv][t] = r
        assert all(set(r[s]) == set(N23_LEVELS[lv][t][s]) for s in SPLITS), f"notebook 23's table of {t} at {lv} lists other machines"
        worst = max([worst] + [abs(r[s][k_] - v) for s in SPLITS for k_, v in N23_LEVELS[lv][t][s].items()]); n_cmp += sum(len(r[s]) for s in SPLITS)
        for p in IDS[t]:
            tr = TRAIN_ROWS[(t, p)]
            LOO_F[(lv, t, p)] = sum(ldn1_left_out_all(frozen_matrix(t, lv, pi, fold_of(p))[tr]) for pi in range(len(FROZEN_PARTS))) / len(FROZEN_PARTS)
        FM.clear(); gc.collect()
    print(f"  {LV_NAME[lv]}: Split A " + ", ".join(f"{t} {FROZEN_RES[lv][t]['A']['avg']:.4f}" for t in TYPES) + f" · {time.time() - t_g0:.0f} s", flush=True)
G0["c"] = {"ok": bool(worst <= G0C_TOL), "values_compared": int(n_cmp), "worst_abs_difference": float(worst), "tolerance": G0C_TOL}
print(f"G0(c) · the frozen MIC system recomputed from the saved embeddings: {n_cmp} values of notebook 23's level tables, largest |Δ| {worst:.1e} "
      f"(tolerance {G0C_TOL}): {G0['c']['ok']}")
if not G0["c"]["ok"]:
    raise SystemExit("stop: G0(c) failed. The saved embeddings do not give notebook 23's level tables")

In [ ]:
# 4 · The audio: microphone 0 of every MIMII clip at a level, in notebook 20's clip order (gate G0(b)), and DCASE 2020's additional
# training clips. Nothing is downloaded here: cells 7 and 9 ask for a level only when its audio is still needed.
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res


def to_int16(v):
    # only for microphone-0 samples read from the 16-bit WAV files, so this conversion is exact
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


STORE = WORK / "stores26"; STORE.mkdir(exist_ok=True)
ST, XS, XMETA, PRE, G0B, PREP_S = {t: {} for t in TYPES}, {}, {}, {}, {}, {}
STOP_PREFETCH = threading.Event()
GATE_F = OUT / "data_gate.json"                     # gate G0(b) as it was found when a store was built (a later session may not need the audio)


def have_store(t, lv):
    return (STORE / f"{t}_CH0_{LEVEL_CODE[lv]}.json").exists() and (STORE / f"{t}_CH0_{LEVEL_CODE[lv]}.npy").exists()


def prefetch(pairs):
    # the zips of `pairs` (type, level), one after another in one background thread, each only if the local disk has room
    pairs = [(t, lv) for t, lv in pairs if not have_store(t, lv)]

    def job():
        for t, lv in pairs:
            if STOP_PREFETCH.is_set():
                return
            if shutil.disk_usage(WORK).free / 1e9 < GBZ[t] * 1.05 + FREE_MARGIN_GB:
                print(f"  (not enough free disk to download {t} at {LV_NAME[lv]} in the background; it will download when its turn comes)", flush=True)
                return
            try:
                get_file(ZIPS[f"{t}|{lv}"])
            except BaseException as e:
                print(f"  (background download of {t} at {LV_NAME[lv]} failed: {repr(e)[:150]}; it will be retried)", flush=True)
                return
    th = threading.Thread(target=job, daemon=True); th.start()
    for k in pairs:
        PRE[k] = th
    return th


def record_gate(key, info):
    d = json.loads(GATE_F.read_text()) if GATE_F.exists() else {}
    d[key] = info; GATE_F.write_text(json.dumps(d, indent=1))


def build_store(t, lv):
    # microphone 0 of the type's clips at a level (int16, exact), in notebook 20's clip order
    code = LEVEL_CODE[lv]; done, arr = STORE / f"{t}_CH0_{code}.json", STORE / f"{t}_CH0_{code}.npy"
    if not (done.exists() and arr.exists()):
        names, t0 = META[t][0], time.time()
        if (t, lv) in PRE:
            PRE[(t, lv)].join()
        zf = zipfile.ZipFile(get_file(ZIPS[f"{t}|{lv}"]))
        ok = np.array_equal(np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav"))), names)
        print(f"G0(b) · {t} at {LV_NAME[lv]}: the zip has Zenodo's MD5, and its clips are notebook 20's: {ok}", flush=True)
        if not ok:
            raise SystemExit(f"stop: G0(b) failed. The clips of {t} at {LV_NAME[lv]} are not notebook 20's list")
        n = len(names); ns = read_all(zf.read(names[0]))[0].shape[0]
        if not TEST:
            assert ns == 10 * SR, (t, lv, ns)
        tmp = STORE / f"{t}_CH0_{code}.part.npy"
        mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int16, shape=(n, ns))
        for i, (x, sr) in enumerate(stream(lambda i: read_all(zf.read(names[i])), range(n))):
            assert sr == SR and x.ndim == 2 and x.shape == (ns, 8), (lv, names[i], sr, x.shape)
            mm[i] = to_int16(np.ascontiguousarray(x[:, 0]))
            if i % 1500 == 0:
                print(f"  {t} at {LV_NAME[lv]}: {i}/{n} clips · {time.time() - t0:.0f} s", flush=True)
        mm.flush(); del mm
        zf.close()
        os.replace(tmp, arr)
        info = {"clips": int(n), "samples": int(ns), "names_are_notebook_20s": bool(ok), "zip": list(ZIPS[f"{t}|{lv}"][1:]), "seconds": round(time.time() - t0, 1)}
        done.write_text(json.dumps(info)); record_gate(f"{t}|{lv}", info)
        if IN_COLAB:
            (WORK / ZIPS[f"{t}|{lv}"][1]).unlink(missing_ok=True)
    G0B[f"{t}|{lv}"] = json.loads(done.read_text())
    ST[t][lv] = np.load(arr, mmap_mode="r")


def level_audio(lv):
    # the four types' stores at one level; the next type's zip downloads in the background while this one is read
    t0 = time.time()
    for th in set(PRE.values()):
        if th.is_alive() and any(k[1] == lv for k, v in PRE.items() if v is th):
            th.join()                                                 # a background download of this level is still running: wait for it
    for j, t in enumerate(TYPES):
        if j + 1 < len(TYPES) and not have_store(TYPES[j + 1], lv) and not (WORK / ZIPS[f"{TYPES[j + 1]}|{lv}"][1]).exists():
            prefetch([(TYPES[j + 1], lv)])
        build_store(t, lv)
    PREP_S[lv] = round(PREP_S.get(lv, 0.0) + time.time() - t0, 1)
    print(f"audio at {LV_NAME[lv]} ready · {(time.time() - t0) / 60:.1f} min", flush=True)


def xc_audio():
    # DCASE 2020's additional machines: every additional-training clip of each type (normal clips; each is microphone 0 at −6 dB)
    t0 = time.time()
    for t in TYPES:
        meta_f, arr = STORE / f"xc_{t}.json", STORE / f"xc_{t}_CH0.npy"
        if not (meta_f.exists() and arr.exists()):
            zf = zipfile.ZipFile(get_file(XZIPS[t]))
            names = np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav")))
            base = [n.replace("\\", "/").split("/")[-1] for n in names]
            mid = np.array([re.search(r"(id_\d+)", b).group(1) for b in base])
            assert all(b.startswith("normal") for b in base), "stop: the additional training data should be normal clips only"
            assert not (set(mid.tolist()) & set(META[t][1].tolist())), "stop: an additional machine has a MIMII machine's ID"
            ns = read_all(zf.read(names[0]))[0].shape[0]
            tmp = STORE / f"xc_{t}_CH0.part.npy"
            mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int16, shape=(len(names), ns))
            for i, (x, sr) in enumerate(stream(lambda i: read_all(zf.read(names[i])), range(len(names)))):
                assert sr == SR and x.shape[0] == ns, (names[i], sr, x.shape)
                mm[i] = to_int16(np.ascontiguousarray(x[:, 0]))
            mm.flush(); del mm
            zf.close()
            os.replace(tmp, arr)
            info = {"names": names.tolist(), "mid": mid.tolist(), "samples": int(ns), "zip": list(XZIPS[t][1:])}
            meta_f.write_text(json.dumps(info)); record_gate(f"{t}|additional", {"clips": int(len(names)), "samples": int(ns), "zip": list(XZIPS[t][1:])})
            if IN_COLAB:
                (WORK / XZIPS[t][1]).unlink(missing_ok=True)
        XMETA[t] = json.loads(meta_f.read_text()); XS[t] = np.load(arr, mmap_mode="r")
        G0B[f"{t}|additional"] = {"clips": len(XMETA[t]["names"]), "samples": XMETA[t]["samples"], "zip": XMETA[t]["zip"]}
    PREP_S["additional"] = round(PREP_S.get("additional", 0.0) + time.time() - t0, 1)
    print("DCASE 2020's additional training clips ready:", {t: len(XMETA[t]["names"]) for t in TYPES}, f"· {time.time() - t0:.0f} s", flush=True)


print("local stores from an earlier run in this session:", [f"{t}|{lv}" for t in TYPES for lv in LVL if have_store(t, lv)] or "none")

In [ ]:
# 5 · The second view: TASTgram-MFN trained with Noisy-ArcMix (Choi and Choi, ICASSP 2024, arXiv 2310.06364)
#
# The model classes, the log-mel settings and the Noisy-ArcMix step are adapted from https://github.com/soonhyeon/Noisy-ArcMix at
# commit 674c759bfaa7723619a4b1d8d00eb9d155c50ba6 (model/net.py, losses.py, dataloader.py, utils.py, trainer.py). Its model file
# says: "modification made on the basis of link:https://github.com/liuyoude/STgram-MFN". Its licence:
#
#   MIT License
#
#   Copyright (c) 2023 soonhyeon
#
#   Permission is hereby granted, free of charge, to any person obtaining a copy
#   of this software and associated documentation files (the "Software"), to deal
#   in the Software without restriction, including without limitation the rights
#   to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
#   copies of the Software, and to permit persons to whom the Software is
#   furnished to do so, subject to the following conditions:
#
#   The above copyright notice and this permission notice shall be included in all
#   copies or substantial portions of the Software.
#
#   THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
#   IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
#   FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
#   AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
#   LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
#   OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
#   SOFTWARE.
#
# Changed here (the rules list every deviation): the number of classes; the frame count and the last kernel follow the clip length
# (313 and 8 × 20 for DCASE's 10 s clips, as in the reference); mixed precision with a gradient scaler, and the ArcFace layer in
# float32 with a floor under its square root; batches come from one int16 tensor and the log-mel is computed on the device; the
# seeds, the hold-out draw, saving and resuming are this notebook's. The reference's unused options (sub-centres, easy margin,
# the other training modes) are left out.
class Bottleneck(nn.Module):
    def __init__(self, inp, oup, stride, expansion):
        super(Bottleneck, self).__init__()
        self.connect = stride == 1 and inp == oup
        self.conv = nn.Sequential(
            # pw
            nn.Conv2d(inp, inp * expansion, 1, 1, 0, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            # dw
            nn.Conv2d(inp * expansion, inp * expansion, 3, stride, 1, groups=inp * expansion, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            # pw-linear
            nn.Conv2d(inp * expansion, oup, 1, 1, 0, bias=False),
            nn.BatchNorm2d(oup),
        )

    def forward(self, x):
        if self.connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class ConvBlock(nn.Module):
    def __init__(self, inp, oup, k, s, p, dw=False, linear=False):
        super(ConvBlock, self).__init__()
        self.linear = linear
        if dw:
            self.conv = nn.Conv2d(inp, oup, k, s, p, groups=inp, bias=False)
        else:
            self.conv = nn.Conv2d(inp, oup, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(oup)
        if not linear:
            self.prelu = nn.PReLU(oup)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        if self.linear:
            return x
        else:
            return self.prelu(x)


Mobilefacenet_bottleneck_setting = [
    # t, c , n ,s
    [2, 128, 2, 2],
    [4, 128, 2, 2],
    [4, 128, 2, 2],
]


class MobileFaceNet(nn.Module):
    def __init__(self, num_class, last_kernel, bottleneck_setting=Mobilefacenet_bottleneck_setting):
        super(MobileFaceNet, self).__init__()
        self.conv1 = ConvBlock(3, 64, 3, 2, 1)
        self.dw_conv1 = ConvBlock(64, 64, 3, 1, 1, dw=True)
        self.inplanes = 64
        block = Bottleneck
        self.blocks = self._make_layer(block, bottleneck_setting)
        self.conv2 = ConvBlock(bottleneck_setting[-1][1], 512, 1, 1, 0)
        self.linear7 = ConvBlock(512, 512, last_kernel, 1, 0, dw=True, linear=True)   # (8, 20) in the reference: 128 × 313 → 8 × 20
        self.linear1 = ConvBlock(512, 128, 1, 1, 0, linear=True)
        self.fc_out = nn.Linear(128, num_class)                       # the reference has it too; with ArcFace on, its output is unused
        # init
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, setting):
        layers = []
        for t, c, n, s in setting:
            for i in range(n):
                if i == 0:
                    layers.append(block(self.inplanes, c, s, t))
                else:
                    layers.append(block(self.inplanes, c, 1, t))
                self.inplanes = c
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.dw_conv1(x)
        x = self.blocks(x)
        x = self.conv2(x)
        x = self.linear7(x)
        x = self.linear1(x)
        return x.reshape(x.size(0), -1)                               # the 128-d feature


class TgramNet(nn.Module):
    def __init__(self, n_frames, num_layer=3, mel_bins=128, win_len=1024, hop_len=512):
        super(TgramNet, self).__init__()
        # if "center=True" of stft, padding = win_len / 2
        self.conv_extrctor = nn.Conv1d(1, mel_bins, win_len, hop_len, win_len // 2, bias=False)
        self.conv_encoder = nn.Sequential(
            *[nn.Sequential(
                nn.LayerNorm(n_frames),                               # 313 in the reference (10 s)
                nn.LeakyReLU(0.2, inplace=True),
                nn.Conv1d(mel_bins, mel_bins, 3, 1, 1, bias=False),
            ) for _ in range(num_layer)])

    def forward(self, x):
        out = self.conv_extrctor(x)
        out = self.conv_encoder(out)
        return out


class Temporal_Attention(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.feature_dim = feature_dim
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 1, 128, 313)
        x = x.squeeze(1)
        x = x.transpose(1, 2)  # (B, 313, 128)
        x1 = self.max_pool(x)  # (B, 313, 1)
        x2 = self.avg_pool(x)  # (B, 313, 1)
        feats = x1 + x2
        feats = feats.repeat(1, 1, self.feature_dim)
        refined_feats = self.sigmoid(feats).transpose(1, 2) * x.transpose(1, 2)
        return refined_feats


class ArcMarginProduct(nn.Module):
    def __init__(self, in_features=128, out_features=200, s=30.0, m=0.7):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = Parameter(torch.Tensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        # make the function cos(theta+m) monotonic decreasing while theta in [0°,180°]
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, x, label):
        cosine = F.linear(F.normalize(x), F.normalize(self.weight))
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp_min(SV["sine_floor"]))   # the floor is this notebook's (a deviation)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where((cosine - self.th) > 0, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=x.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output = output * self.s
        return output


MEMFMT = torch.contiguous_format                                      # set by pick_memory_format (a speed setting only)


class TASTgramMFN(nn.Module):
    def __init__(self, num_classes, n_frames, last_kernel, c_dim=128, win_len=1024, hop_len=512,
                 bottleneck_setting=Mobilefacenet_bottleneck_setting, m=0.7, s=30):
        super().__init__()
        self.arcface = ArcMarginProduct(in_features=c_dim, out_features=num_classes, m=m, s=s)
        self.tgramnet = TgramNet(n_frames, mel_bins=c_dim, win_len=win_len, hop_len=hop_len)
        self.mobilefacenet = MobileFaceNet(num_class=num_classes, last_kernel=last_kernel, bottleneck_setting=bottleneck_setting)
        self.temporal_attention = Temporal_Attention(feature_dim=c_dim)

    def forward(self, x_wav, x_mel, label):
        # the reference's "noisy_arcmix" mode: the margin goes to `label` (in training, the first clip's class)
        with torch.autocast(device_type=DEVICE, enabled=AMP):
            x_t = self.tgramnet(x_wav).unsqueeze(1)
            x_mel_temp_att = self.temporal_attention(x_mel).unsqueeze(1)
            x = torch.cat((x_t.float(), x_mel, x_mel_temp_att), dim=1)   # .float(): the three channels in one type under autocast
            feature = self.mobilefacenet(x.contiguous(memory_format=MEMFMT))
        feature = feature.float()
        with torch.autocast(device_type=DEVICE, enabled=False):      # ArcFace in float32 (a deviation)
            out = self.arcface(feature, label)
        return out, feature


MEL = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=SV["n_fft"], hop_length=SV["hop"], n_mels=SV["n_mels"], power=2,
                                           pad_mode="constant", norm="slaney", mel_scale="slaney").to(DEVICE)
CE = nn.CrossEntropyLoss()


@torch.no_grad()
def log_mel(x):
    # (B, samples) float32 in [−1, 1] → (B, 1, mel bands, frames): the reference's file_to_log_mel_spectrogram with power = 2
    return (20.0 / 2 * torch.log10(MEL(x) + sys.float_info.epsilon)).unsqueeze(1)


def frames_of(ns):
    # the frames of an STFT with center=True, which is also the length of the Tgram convolution (kernel 1024, stride 512, padding 512)
    return 1 + ns // SV["hop"]


def last_kernel(n_mels, n_frames):
    # the feature map after the network's four stride-2 stages (3 × 3, padding 1): 128 × 313 → 8 × 20
    h, w = n_mels, n_frames
    for _ in range(4):
        h, w = (h - 1) // 2 + 1, (w - 1) // 2 + 1
    return (h, w)


def training_lists():
    # one class per machine, 28 in all: for each type its four MIMII machines (their normal training clips in the program's split,
    # microphone 0 at −6 dB), then DCASE 2020's three additional machines (all their additional-training clips);
    # items (type index, source: 0 = MIMII at −6 dB / 1 = additional, row in that store, class)
    items, classes = [], []
    for ti, t in enumerate(TYPES):
        for p in IDS[t]:
            c = len(classes); classes.append(f"{t}|{p}|MIMII"); items.extend((ti, 0, int(r), c) for r in TRAIN_ROWS[(t, p)])
        xm = np.array(XMETA[t]["mid"])
        for q in sorted(set(xm.tolist())):
            c = len(classes); classes.append(f"{t}|{q}|DCASE2020-additional"); items.extend((ti, 1, int(r), c) for r in np.where(xm == q)[0])
    return np.array(items, dtype=np.int64), classes


AUDIO, LABELS, STORE_DEV, MF_TIMES = None, None, None, {}


def build_audio():
    # every training-list clip in one int16 tensor: on the GPU when it has room (batches are then gathered without the CPU), else in RAM
    global AUDIO, LABELS, STORE_DEV
    if AUDIO is not None:
        return
    n = len(ITEMS); need = n * NS * 2 / 1e9; dev = DEVICE; t0 = time.time()
    if DEVICE == "cuda" and torch.cuda.mem_get_info()[0] / 1e9 < need + SV["gpu_headroom_gb"]:
        dev = "cpu"
    A = torch.empty((n, NS), dtype=torch.int16, device=dev)
    for ti, t in enumerate(TYPES):
        for source, mm in ((0, ST[t][DECISION_LEVEL]), (1, XS[t])):
            pos = np.where((ITEMS[:, 0] == ti) & (ITEMS[:, 1] == source))[0]
            for i in range(0, len(pos), 256):
                p = pos[i:i + 256]
                A[torch.from_numpy(p).to(dev)] = torch.from_numpy(np.ascontiguousarray(mm[ITEMS[p, 2]])).to(dev)
    AUDIO, STORE_DEV, LABELS = A, dev, torch.from_numpy(ITEMS[:, 3].copy()).to(dev)
    print(f"training audio: {n} clips, {need:.2f} GB as int16, kept on {'the GPU' if dev == 'cuda' else 'the CPU'} · {time.time() - t0:.0f} s", flush=True)


def fetch(ids):
    # clips `ids` of the training lists → float32 waves in [−1, 1] and their classes, on the device
    j = torch.from_numpy(np.ascontiguousarray(ids)).to(AUDIO.device)
    return AUDIO[j].to(DEVICE).float() / 32768.0, LABELS[j].to(DEVICE)


def build_net(seed):
    torch.manual_seed(seed)
    return TASTgramMFN(len(CLASSES), N_FRAMES, LAST_KERNEL, c_dim=SV["c_dim"], win_len=SV["win_len"], hop_len=SV["hop"], m=SV["arc_m"], s=SV["arc_s"])


def mixed_step(net, x, y, lam, perm):
    # the reference's mixup_data and noisy_arcmix_criterion: the waves and the log-mels are mixed with one λ and one permutation;
    # the margin is at the first clip's class; the loss is λ·CE(first class) + (1 − λ)·CE(second class) on the same logits
    mel = log_mel(x)
    logits, _ = net((lam * x + (1 - lam) * x[perm]).unsqueeze(1), lam * mel + (1 - lam) * mel[perm], y)
    return lam * CE(logits, y) + (1 - lam) * CE(logits, y[perm]), logits


def pick_memory_format():
    # the same mathematics in two memory layouts: a few training steps are timed in each on this GPU, and the faster one is kept.
    # These are also the first real training steps: if neither layout runs with the audio on the GPU (most likely the GPU's memory
    # is too small for both), the audio is moved to the CPU's memory and the steps are tried once more.
    global MEMFMT, AUDIO, LABELS, STORE_DEV
    if DEVICE != "cuda":
        return
    err = ""
    for attempt in (0, 1):
        MF_TIMES.clear()
        x, y = fetch(np.arange(min(SV["batch"], len(ITEMS)))); perm = torch.arange(len(y) - 1, -1, -1, device=DEVICE)
        for name, fmt in (("contiguous", torch.contiguous_format), ("channels_last", torch.channels_last)):
            MEMFMT = fmt; net = opt = scaler = loss = None
            try:
                net = build_net(0).to(DEVICE).to(memory_format=fmt); net.train()
                opt = torch.optim.AdamW(net.parameters(), lr=SV["lr"]); scaler = torch.amp.GradScaler("cuda", enabled=AMP)
                for i in range(16):
                    if i == 4:
                        torch.cuda.synchronize(); t0 = time.time()
                    loss, _ = mixed_step(net, x, y, 0.5, perm)
                    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                torch.cuda.synchronize(); MF_TIMES[name] = round((time.time() - t0) / 12, 4)
            except Exception as e:
                MF_TIMES[name] = None; err = repr(e)[:160]
                print(f"  ({name} could not be timed: {err})", flush=True)
            del net, opt, scaler, loss
            gc.collect(); torch.cuda.empty_cache()
        ok = {k: v for k, v in MF_TIMES.items() if v}
        del x, y, perm
        if ok or STORE_DEV != "cuda":
            break
        AUDIO, LABELS, STORE_DEV = AUDIO.cpu(), LABELS.cpu(), "cpu"
        gc.collect(); torch.cuda.empty_cache()
        print("  no training step ran with the audio on the GPU: the audio is moved to the CPU's memory, and the steps are tried again", flush=True)
    if not ok:
        raise SystemExit(f"stop: a training step does not run on this GPU ({err}). Choose G4 in Runtime › Change runtime type, then Run all.")
    best = min(ok, key=ok.get)
    MEMFMT = torch.channels_last if best == "channels_last" else torch.contiguous_format
    print(f"memory layout: seconds per training step {MF_TIMES} → {best}", flush=True)


@torch.no_grad()
def hold_out_pass(net, hold):
    # as the reference's valid(): evaluation mode, margin at the clip's class, no mixing; the mean of the batches' means
    net.eval(); losses, accs = [], []
    for i in range(0, len(hold), SV["batch"]):
        x, y = fetch(hold[i:i + SV["batch"]])
        logits, _ = net(x.unsqueeze(1), log_mel(x), y)
        losses.append(float(CE(logits, y))); accs.append(float((logits.argmax(1) == y).float().mean()))
    return float(np.mean(losses)), float(np.mean(accs))


def fmt_time(s):
    return f"{s / 3600:.1f} h" if s >= 5400 else f"{s / 60:.0f} min"


def hold_out_split():
    # 5 % held out at random, one draw for the three seeds (as notebook 25): the held-out clips are then unseen by every seed's model.
    # The draw is saved in Drive the first time and read from there afterwards, so that a later session uses the same clips.
    n = len(ITEMS); n_tr = int(n * SV["split_ratio"]); f = OUT / "hold_out.json"
    if f.exists():
        d = json.loads(f.read_text())
        assert d["n_clips"] == n and len(d["hold_out"]) == n - n_tr, "the saved hold-out belongs to other training lists"
        hold = np.array(d["hold_out"], dtype=np.int64)
    else:
        hold = np.sort(np.random.default_rng([26, 0]).permutation(n)[n_tr:])
        f.write_text(json.dumps({"n_clips": int(n), "hold_out": hold.tolist()}))
    return np.setdiff1d(np.arange(n), hold), hold


def train_sv(seed):
    # one training of the second view: 5 % held out, Noisy-ArcMix on every batch, AdamW with cosine annealing per epoch, the model
    # with the lowest hold-out loss kept (as the reference); saved to Drive every SV["save_every"] epochs and resumed from there
    fin, res = OUT / f"sv_s{seed}.pt", OUT / f"sv_s{seed}_resume.pt"
    build_audio()
    tr_idx, hold = hold_out_split(); n, n_tr = len(ITEMS), len(tr_idx)
    hold_md5 = hashlib.md5(hold.astype("<i8").tobytes()).hexdigest()
    bs, epochs = SV["batch"], SV["epochs"]; steps = math.ceil(n_tr / bs)
    net = build_net(seed).to(DEVICE).to(memory_format=MEMFMT)
    opt = torch.optim.AdamW(net.parameters(), lr=SV["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=SV["eta_min_ratio"] * SV["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log = {"seed": seed, "cfg": dict(SV), "classes": CLASSES, "n_clips": int(n), "n_train": int(n_tr), "n_hold_out": int(n - n_tr),
           "hold_out_md5": hold_md5, "steps_per_epoch": int(steps), "n_frames": int(N_FRAMES), "last_kernel": list(LAST_KERNEL),
           "parameters": int(sum(p.numel() for p in net.parameters())), "epochs": [], "resumed_at_epoch": [], "sessions": []}
    best, start = {"val_loss": float("inf"), "epoch": -1, "state": None}, 0
    if res.exists():
        st = torch.load(res, map_location="cpu", weights_only=True)
        log = json.loads(st["log"])
        assert log["classes"] == CLASSES and log["hold_out_md5"] == hold_md5 and log["cfg"] == json.loads(json.dumps(SV)), \
            "the saved training used other lists or settings"
        net.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        best = {"val_loss": st["best_val_loss"], "epoch": st["best_epoch"], "state": st["best_state"]}
        start = st["epoch"] + 1; log["resumed_at_epoch"].append(start)
        print(f"seed {seed}: resuming after epoch {start}", flush=True)
    session = {"gpu": GPU, "memory_layout": "channels_last" if MEMFMT == torch.channels_last else "contiguous", "audio_on": STORE_DEV,
               "from_epoch": start, "step_seconds": None}
    log["sessions"].append(session)
    print(f"seed {seed}: {n_tr} training clips and {n - n_tr} held out · {len(CLASSES)} classes · {steps} steps per epoch · "
          f"{log['parameters']:,} parameters", flush=True)
    t_from, t_to = min(5, steps - 2), min(45, steps - 1)
    for ep in range(start, epochs):
        rng = np.random.default_rng([seed, 26, 1, ep]); order = tr_idx[rng.permutation(n_tr)]
        lr_now = opt.param_groups[0]["lr"]
        net.train(); t0, tot, acc = time.time(), 0.0, 0.0
        for b in range(steps):
            x, y = fetch(order[b * bs:(b + 1) * bs])
            lam = float(rng.beta(SV["alpha"], SV["alpha"])); pb = torch.from_numpy(rng.permutation(len(y))).to(DEVICE)
            loss, logits = mixed_step(net, x, y, lam, pb)
            if not bool(torch.isfinite(loss)):
                raise SystemExit(f"stop: G1 failed. The training loss of seed {seed} is not finite (epoch {ep + 1}, step {b + 1})")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot += float(loss.detach()); acc += float((logits.detach().argmax(1) == y).float().mean())
            if ep == start and b in (t_from, t_to) and t_to > t_from >= 0:
                if DEVICE == "cuda":
                    torch.cuda.synchronize()
                if b == t_from:
                    t_mark = time.time()
                else:
                    session["step_seconds"] = round((time.time() - t_mark) / (t_to - t_from), 4)
                    left_here = (epochs - ep) * steps * session["step_seconds"] * 1.03
                    later = [s for s in SV["seeds"] if s != seed and not (OUT / f"sv_s{s}.pt").exists()]
                    left_all = left_here + len(later) * epochs * steps * session["step_seconds"] * 1.03
                    print(f"  this GPU ({GPU or 'CPU'}): {session['step_seconds']:.3f} s per step → about {fmt_time(left_here)} left for this "
                          f"training, about {fmt_time(left_all)} for all the trainings that are left", flush=True)
                    if left_all > 6 * 3600 and not TEST:
                        print("  NOTE: that is a long run on this GPU. It can stay as it is (it resumes after a disconnect). For a much shorter "
                              "run: Runtime › Change runtime type › G4, then Runtime › Run all; what is saved in Drive is kept.", flush=True)
        sched.step()
        vl, va = hold_out_pass(net, hold)
        if not math.isfinite(vl):
            raise SystemExit(f"stop: G1 failed. The hold-out loss of seed {seed} is not finite (epoch {ep + 1})")
        if best["val_loss"] > vl:                                     # the reference saves the model whenever the hold-out loss falls
            best = {"val_loss": vl, "epoch": ep, "state": {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}}
        log["epochs"].append({"epoch": ep, "train_loss": round(tot / steps, 5), "train_accuracy": round(acc / steps, 5), "hold_out_loss": round(vl, 5),
                              "hold_out_accuracy": round(va, 5), "lr": lr_now, "seconds": round(time.time() - t0, 1)})
        if ep < start + 3 or (ep + 1) % 5 == 0 or ep + 1 == epochs:
            print(f"seed {seed} · epoch {ep + 1}/{epochs} · loss {tot / steps:.4f} · hold-out loss {vl:.4f}, accuracy {va:.4f} · "
                  f"best epoch {best['epoch'] + 1} · {time.time() - t0:.0f} s", flush=True)
        if (ep + 1) % SV["save_every"] == 0 and ep + 1 < epochs:
            tmp = res.with_name(res.name + ".tmp")
            torch.save({"model": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "epoch": ep, "best_state": best["state"], "best_epoch": best["epoch"],
                        "best_val_loss": best["val_loss"], "log": json.dumps(log)}, tmp)
            os.replace(tmp, res)
    log["best_epoch"], log["best_hold_out_loss"] = int(best["epoch"]), float(best["val_loss"])
    tmp = fin.with_name(fin.name + ".tmp")
    torch.save({"state": best["state"], "last_state": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "best_epoch": int(best["epoch"]),
                "best_val_loss": float(best["val_loss"]), "classes": CLASSES, "hold_out": hold.tolist(), "log": json.dumps(log)}, tmp)
    os.replace(tmp, fin)
    res.unlink(missing_ok=True)
    del net, opt, sched, scaler
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def feat_file(seed, lv):
    return OUT / f"emb_SV_s{seed}_{LEVEL_CODE[lv]}.npz"


@torch.no_grad()
def embed_sv(seed, lv):
    # the saved model on every MIMII clip at one level: the 128-d feature, the paper's score (the cross-entropy for the clip's
    # own machine class, margin on, as the reference's test), and whether the largest logit is the clip's own machine
    fin = OUT / f"sv_s{seed}.pt"
    st = torch.load(fin, map_location="cpu", weights_only=True)
    assert st["classes"] == CLASSES, "the saved model used other classes"
    net = build_net(seed); net.load_state_dict(st["state"]); net.to(DEVICE).to(memory_format=MEMFMT).eval()
    out, bs = {}, SV["embed_batch"]
    for t in TYPES:
        mm = ST[t][lv]; n = mm.shape[0]
        assert n == len(META[t][0]) and mm.shape[1] == NS, f"the store of {t} at {lv} is not this notebook's"
        y_all = np.array([CLS_OF[(t, q)] for q in META[t][1]], dtype=np.int64)
        V, ce, hit = np.zeros((n, SV["c_dim"]), np.float32), np.zeros(n, np.float32), np.zeros(n, np.uint8)
        for i in range(0, n, bs):
            x = torch.from_numpy(f32(mm[i:i + bs])).to(DEVICE); y = torch.from_numpy(y_all[i:i + bs]).to(DEVICE)
            logits, feat = net(x.unsqueeze(1), log_mel(x), y)
            V[i:i + len(y)] = feat.cpu().numpy(); ce[i:i + len(y)] = F.cross_entropy(logits, y, reduction="none").cpu().numpy()
            hit[i:i + len(y)] = (logits.argmax(1) == y).cpu().numpy()
        assert np.isfinite(V).all() and np.isfinite(ce).all(), f"non-finite features for {t} at {lv}"
        out[f"V__{t}"], out[f"CE__{t}"], out[f"HIT__{t}"] = V, ce, hit
    tmp = OUT / f"emb_SV_s{seed}_{LEVEL_CODE[lv]}.part.npz"
    np.savez(tmp, **out, meta=json.dumps({"seed": seed, "level": lv, "weights_md5": model_md5(seed), "best_epoch": int(st["best_epoch"]), "test_run": TEST,
                                          "date": time.strftime("%Y-%m-%d %H:%M")}))
    os.replace(tmp, feat_file(seed, lv))
    del net
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

In [ ]:
# 6 · How systems are scored on Split A at one level: the frozen system, the view alone, and the fused form ZP with its spreads
VIEWS, CES, HITS, VM, MMD5 = {}, {}, {}, {}, {}                       # (seed, level) → type → features (n, 128), the paper's score (n,), own-class flags
HOLD_ROWS, CAL_P, CAL_FZ, CAL_B, B1_HO = {}, {}, {}, {}, {}           # type → rows (in notebook 20's list) of its held-out MIMII clips; the spreads
SYSTEMS = ["frozen", "view_paper", "view_LDN1", "view_B1", "ZP", "ZB"]   # ZP is the fused system; ZB is a reported reading (the rules)
LABEL = {"frozen": "frozen (notebook 23)", "view_paper": "view alone · paper's score", "view_LDN1": "view alone · LDN1", "view_B1": "view alone · B1",
         "ZP": "ZP · frozen + view", "ZB": "ZB · the view by B1 (reported)", "ZP_m6cal": "ZP · spreads carried from −6 dB"}
NAME = {"pump": "pump", "fan": "fan", "valve": "valve", "slider": "slide rail"}


def model_md5(seed):
    if seed not in MMD5:
        MMD5[seed] = md5(OUT / f"sv_s{seed}.pt")
    return MMD5[seed]


def feats_ok(seed, lv):
    # the seed's features at a level are in Drive, belong to its saved model and hold everything this notebook reads
    fin, ef = OUT / f"sv_s{seed}.pt", feat_file(seed, lv)
    if not (fin.exists() and ef.exists()):
        return False
    D = np.load(ef, allow_pickle=False)
    return bool(json.loads(str(D["meta"]))["weights_md5"] == model_md5(seed) and all(f"{k}__{t}" in D.files for k in ("V", "CE", "HIT") for t in TYPES))


def load_feats(seed, lv):
    D = np.load(feat_file(seed, lv), allow_pickle=False)
    VIEWS[(seed, lv)] = {t: D[f"V__{t}"].astype(np.float64) for t in TYPES}; CES[(seed, lv)] = {t: D[f"CE__{t}"].astype(np.float64) for t in TYPES}
    HITS[(seed, lv)] = {t: D[f"HIT__{t}"].astype(bool) for t in TYPES}
    assert all(len(VIEWS[(seed, lv)][t]) == len(META[t][0]) for t in TYPES), "the saved features do not have notebook 20's number of clips"


def view_matrix(seeds, lv, t):
    # the program's seed ensemble: each seed's features unit-normalised, concatenated, ÷√seeds
    k = (tuple(seeds), lv, t)
    if k not in VM:
        VM[k] = np.concatenate([unit(VIEWS[(s, lv)][t]) for s in seeds], 1) / math.sqrt(len(seeds))
    return VM[k]


def paper(seeds, lv, t, rows):
    # the paper's score of the clips `rows` at a level: the cross-entropy for the clip's own machine class, averaged over the seeds
    return sum(CES[(s, lv)][t][rows] for s in seeds) / len(seeds)


def sigma_F(lv, t):
    # the frozen system's spread at a level: its log score of every normal training clip of the type's machines, each against its
    # machine's other training clips (LOO_F, cell 3). No test clip.
    k = (lv, t)
    if k not in CAL_FZ:
        CAL_FZ[k] = spread(np.log(np.maximum(np.concatenate([LOO_F[(lv, t, p)] for p in IDS[t]]), LDN_FLOOR)))
    return CAL_FZ[k]


def sigma_P(seeds, lv, t):
    # the view's spread at a level: the log of the paper's score on the held-out MIMII clips of the type, which no seed trained on
    k = (tuple(seeds), lv, t)
    if k not in CAL_P:
        rows = np.asarray(HOLD_ROWS[t])
        CAL_P[k] = spread(np.log(np.maximum(paper(seeds, lv, t, rows), LDN_FLOOR)) if len(rows) else np.zeros(0))
    return CAL_P[k]


def b1_left_out(R, pos):
    # B1 of the reference clips at positions `pos`, each against all the other references (the clip itself left out)
    U = unit(R); S = U @ U.T; np.fill_diagonal(S, -np.inf)
    return 1.0 - S[np.asarray(pos, dtype=np.int64)].max(1)


_R6 = np.random.default_rng(6).standard_normal((30, 12))
assert np.allclose(b1_left_out(_R6, [3, 17]), [b1(np.delete(_R6, h, 0), _R6[[h]])[0] for h in (3, 17)]), "stop: the left-out B1 is not B1"


def b1_held_out(seeds, lv, t):
    # the view's B1 of the type's held-out clips at a level, each against its machine's other training clips there (the references a
    # test clip meets, less the clip itself): row in the type's clip list → score. A machine with fewer than three training clips has none.
    k = (tuple(seeds), lv, t)
    if k not in B1_HO:
        V, d = view_matrix(seeds, lv, t), {}
        for p in IDS[t]:
            rows = np.asarray(TRAIN_ROWS[(t, p)]); pos = np.where(np.isin(rows, HOLD_ROWS[t]))[0]
            if len(pos) and len(rows) >= 3:
                d.update(zip(rows[pos].tolist(), b1_left_out(V[rows], pos).tolist()))
        B1_HO[k] = d
    return B1_HO[k]


def sigma_B(seeds, lv, t):
    # the spread for the reported reading ZB: the log of that B1 over the held-out clips. No test clip.
    k = (tuple(seeds), lv, t)
    if k not in CAL_B:
        CAL_B[k] = spread(np.log(np.maximum(np.array(list(b1_held_out(seeds, lv, t).values()), dtype=np.float64), LDN_FLOOR)))
    return CAL_B[k]


def zb_ok(seeds, lv):
    return all(c["n"] >= CAL_MIN and c["sigma"] is not None and c["sigma"] > 1e-9 for t in TYPES for c in (sigma_F(lv, t), sigma_B(seeds, lv, t)))


def zp_ok(seeds, lv):
    # ZP needs both spreads, each from at least CAL_MIN clips and not zero, for every type
    return all(c["n"] >= CAL_MIN and c["sigma"] is not None and c["sigma"] > 1e-9 for t in TYPES for c in (sigma_F(lv, t), sigma_P(seeds, lv, t)))


def score_of(name, seeds, lv, t, p):
    # one system's anomaly scores of machine p's test clips at a level (Split A: the references are its normal training clips there)
    tr, te = SPLIT[t][p]
    if name == "frozen":
        return SF[(lv, t, p)]
    if name == "view_paper":
        return paper(seeds, lv, t, te)
    if name in ("ZP", "ZP_m6cal"):                                    # each system's log score in units of its spread on normal clips
        cl = DECISION_LEVEL if name == "ZP_m6cal" else lv             # the level the spreads are measured at
        return (np.log(np.maximum(SF[(lv, t, p)], LDN_FLOOR)) / sigma_F(cl, t)["sigma"]
                + np.log(np.maximum(paper(seeds, lv, t, te), LDN_FLOOR)) / sigma_P(seeds, cl, t)["sigma"])
    V = view_matrix(seeds, lv, t)
    if name == "view_LDN1":
        return ldn1(V[tr], V[te])
    if name == "view_B1":
        return b1(V[tr], V[te])
    if name == "ZB":                                                  # reported only: the calibrated sum with the view's B1 in place of the paper's score
        return (np.log(np.maximum(SF[(lv, t, p)], LDN_FLOOR)) / sigma_F(lv, t)["sigma"]
                + np.log(np.maximum(b1(V[tr], V[te]), LDN_FLOOR)) / sigma_B(seeds, lv, t)["sigma"])
    raise ValueError(name)


def table(seeds, lv, names, keep=None):
    # Split A at one level. Per system and type: every machine's AUC and pAUC in full, and the program's table entries (notebook 11:
    # each machine's AUC rounded to four decimals, and the mean of the four, rounded). With `keep`, every test clip's scores are kept.
    names = list(names)
    if any(nm.startswith("ZP") for nm in names) and not zp_ok(seeds, lv):
        raise SystemExit(f"stop: ZP cannot be formed at {LV_NAME[lv]}: a spread rests on fewer than {CAL_MIN} clips or is zero. "
                         f"frozen {[sigma_F(lv, t) for t in TYPES]} · view {[sigma_P(seeds, lv, t) for t in TYPES]}")
    if "ZB" in names and not zb_ok(seeds, lv):                        # a reported reading: left out, not a stop
        names.remove("ZB"); print(f"  (ZB is left out at {LV_NAME[lv]}: a spread for it rests on fewer than {CAL_MIN} clips or is zero)")
    out = {nm: {} for nm in names}
    for t in TYPES:
        label = META[t][2]
        for nm in names:
            per, a4 = {}, {}
            for p in IDS[t]:
                te = SPLIT[t][p][1]; y = label[te]; sc = np.asarray(score_of(nm, seeds, lv, t, p), np.float64)
                assert np.isfinite(sc).all(), f"non-finite scores: {nm}, {t} {p} at {lv}"
                if keep is not None:
                    keep.setdefault((lv, t, p), {"file": [str(META[t][0][i]) for i in te], "label": y})[nm] = sc
                per[p] = {"AUC": float(roc_auc_score(y, sc)), "pAUC": float(roc_auc_score(y, sc, max_fpr=0.1)), "n_test": int(len(y)), "n_anomalies": int(np.sum(y))}
                a4[p] = round(per[p]["AUC"], 4)
            a4["avg"] = round(float(np.mean([a4[p] for p in IDS[t]])), 4)
            out[nm][t] = {"per_machine": per, "A": a4}
        VM.clear(); gc.collect()
    if "frozen" in out:                                               # the frozen column is cell 3's table, which gate G0(c) tied to notebook 23
        assert all(out["frozen"][t]["A"] == FROZEN_RES[lv][t]["A"] for t in TYPES), "the frozen system's Split-A table changed between cells"
    return out


def s2(res):
    # the rules' two scores on Split A, four decimals: development = the mean of pumps and fans; confirmation = of valves and slide rails
    return (round(float(np.mean([res[t]["A"]["avg"] for t in MAIN])), 4), round(float(np.mean([res[t]["A"]["avg"] for t in AUX])), 4))


def g_view(res):
    # gate G-VIEW: for every type, the view alone reaches the DCASE 2020 baseline (the same recordings in DCASE's split) on Split A at
    # −6 dB with the paper's score or with LDN1. It is there to catch a broken training or a broken pass.
    return {t: {"AUC_paper": res["view_paper"][t]["A"]["avg"], "AUC_LDN1": res["view_LDN1"][t]["A"]["avg"], "baseline": DCASE_BASELINE_DEV[t],
                "ok": bool(max(res["view_paper"][t]["A"]["avg"], res["view_LDN1"][t]["A"]["avg"]) >= DCASE_BASELINE_DEV[t])} for t in TYPES}


def show(res, names, title, with_scores=True):
    print(title)
    print(f"  {'':<34}" + "".join(f"{NAME[t]:>12}" for t in TYPES) + (f"{'development':>14}{'confirmation':>14}" if with_scores else ""))
    for nm in [n_ for n_ in names if n_ in res]:
        d, c = s2(res[nm])
        print(f"  {LABEL[nm]:<34}" + "".join(f"{res[nm][t]['A']['avg']:>12.4f}" for t in TYPES) + (f"{d:>14.4f}{c:>14.4f}" if with_scores else ""))


def carried_over(seeds, lv):
    # a look that needs no abnormal clip: on the held-out MIMII clips at a level (normal clips that no seed trained on), the share the
    # view assigns to their own machine (margin on, mean over the seeds) and the median of their paper's score
    return {t: {"held_out_clips": int(len(HOLD_ROWS[t])),
                "assigned_to_own_machine": float(np.mean([HITS[(s, lv)][t][HOLD_ROWS[t]].mean() for s in seeds])) if len(HOLD_ROWS[t]) else None,
                "median_paper_score": float(np.median(paper(seeds, lv, t, HOLD_ROWS[t]))) if len(HOLD_ROWS[t]) else None} for t in TYPES}


def spreads(seeds, lv):
    return {"frozen": {t: sigma_F(lv, t) for t in TYPES}, "view_paper": {t: sigma_P(seeds, lv, t) for t in TYPES},
            "view_B1": {t: sigma_B(seeds, lv, t) for t in TYPES}}

In [ ]:
# 7 · Train the three seeds (each: training or resume, then its features of every MIMII clip at −6 dB); gate G-VIEW after the first.
# The −6 dB audio is downloaded only if a training or a −6 dB feature file is still missing.
LISTS_F = OUT / "training_lists.json"
TODO = [s for s in SV["seeds"] if not (OUT / f"sv_s{s}.pt").exists()]
NEED_AUDIO = bool(TODO) or not LISTS_F.exists() or not all(feats_ok(s, DECISION_LEVEL) for s in SV["seeds"])
if NEED_AUDIO:
    if not TEST and shutil.disk_usage(WORK).free / 1e9 < DISK_MIN_GB:
        raise SystemExit(f"stop: the local disk has {shutil.disk_usage(WORK).free / 1e9:.0f} GB free and this run needs {DISK_MIN_GB:.0f} GB")
    level_audio(DECISION_LEVEL); xc_audio()


def lists():
    # the training lists: built from this session's audio lists when they are here, and kept in Drive, so that a later session which
    # needs no audio still knows the classes and the held-out clips. The two must agree.
    built = None
    if all(t in XMETA for t in TYPES):
        ns = {int(ST[t][DECISION_LEVEL].shape[1]) for t in TYPES} | {int(XMETA[t]["samples"]) for t in TYPES}
        assert len(ns) == 1, f"the clips differ in length: {ns}"
        items, classes = training_lists(); built = {"classes": classes, "items": items.tolist(), "samples": ns.pop()}
    if LISTS_F.exists():
        d = json.loads(LISTS_F.read_text())
        if built is not None and built != d:
            raise SystemExit("stop: this session's training lists are not the ones saved in Drive › P04_SecondViewMIMII")
    else:
        d = built; LISTS_F.write_text(json.dumps(d))
    return np.array(d["items"], dtype=np.int64), d["classes"], {tuple(c.split("|")[:2]): i for i, c in enumerate(d["classes"])}, int(d["samples"])


ITEMS, CLASSES, CLS_OF, NS = lists()
N_FRAMES = frames_of(NS); LAST_KERNEL = last_kernel(SV["n_mels"], N_FRAMES)
_, HOLD = hold_out_split(); N_TRAIN = len(ITEMS) - len(HOLD)
_h = ITEMS[HOLD]
HOLD_ROWS.update({t: _h[(_h[:, 0] == ti) & (_h[:, 1] == 0)][:, 2] for ti, t in enumerate(TYPES)})
N_MIMII, N_XC = int(np.sum(ITEMS[:, 1] == 0)), int(np.sum(ITEMS[:, 1] == 1))
G0["d"] = {"classes": len(CLASSES), "clips": int(len(ITEMS)), "mimii": N_MIMII, "additional": N_XC, "train": int(N_TRAIN), "held_out": int(len(HOLD)),
           "held_out_mimii_per_type": {t: int(len(HOLD_ROWS[t])) for t in TYPES}, "frames": int(N_FRAMES), "last_kernel": list(LAST_KERNEL),
           "ok": bool((N_FRAMES, LAST_KERNEL, len(CLASSES), len(ITEMS), N_TRAIN, N_MIMII, N_XC) == (REF_FRAMES, REF_KERNEL, REF_CLASSES, REF_CLIPS, REF_TRAIN, REF_MIMII, REF_XC))}
print(f"G0(d) · training lists: {len(ITEMS)} clips ({N_MIMII} MIMII at −6 dB, {N_XC} additional) in {len(CLASSES)} classes · {N_FRAMES} frames · "
      f"last kernel {LAST_KERNEL} · {N_TRAIN} train, {len(HOLD)} held out (the same clips for every seed; MIMII among them: {G0['d']['held_out_mimii_per_type']})"
      f" · the rules' sizes: {G0['d']['ok']}")
if not TEST and not G0["d"]["ok"]:
    raise SystemExit("stop: G0(d) failed, the lists or the sizes are not the rules'")
assert all(CLS_OF[(t, p)] == CLASSES.index(f"{t}|{p}|MIMII") for t in TYPES for p in IDS[t]), "a MIMII machine has no class"
print(f"trainings to do: {len(TODO)} of {len(SV['seeds'])} · {math.ceil(N_TRAIN / SV['batch'])} steps per epoch, {SV['epochs']} epochs each. "
      "Notebook 25 measured 0.019–0.020 s per step on an RTX PRO 6000 (G4): about 31 minutes per training there. The measured estimate follows.")
TH_B = None
if TODO:
    if not all(feats_ok(s, lv) for s in SV["seeds"] for lv in OTHER_LEVELS):
        TH_B = prefetch([(t, lv) for lv in OTHER_LEVELS for t in TYPES])   # the 6 and 0 dB zips download while the trainings run
    build_audio(); pick_memory_format()
LOGS, G1, T_TRAIN, T_EMBED, PREVIEW, GV0 = {}, {}, {}, {}, None, None


def ensure_feats(seed, lv):
    if not feats_ok(seed, lv):
        t0 = time.time(); embed_sv(seed, lv); T_EMBED[f"{seed}|{lv}"] = round(time.time() - t0, 1)
        print(f"seed {seed}: features of every MIMII clip at {LV_NAME[lv]} saved · {T_EMBED[f'{seed}|{lv}']:.0f} s", flush=True)
    load_feats(seed, lv)


def ensure_sv(seed):
    # one seed: trained (or resumed, or read from Drive), then its features of every MIMII clip at −6 dB; gate G1
    fin = OUT / f"sv_s{seed}.pt"
    if not fin.exists():
        t0 = time.time(); train_sv(seed); T_TRAIN[seed] = round(time.time() - t0, 1)
    ensure_feats(seed, DECISION_LEVEL)
    st = torch.load(fin, map_location="cpu", weights_only=True)
    assert st["hold_out"] == HOLD.tolist() and st["classes"] == CLASSES, "the saved model was trained with another hold-out or other classes"
    log = json.loads(st["log"]); LOGS[seed] = log
    e = log["epochs"]
    G1[seed] = {"first_train_loss": e[0]["train_loss"], "last_train_loss": e[-1]["train_loss"], "first_hold_out_loss": e[0]["hold_out_loss"],
                "best_hold_out_loss": round(log["best_hold_out_loss"], 5), "best_epoch": log["best_epoch"] + 1,
                "hold_out_accuracy_at_best": e[log["best_epoch"]]["hold_out_accuracy"], "epochs": len(e),
                "ok": bool(len(e) == SV["epochs"] and e[-1]["train_loss"] < e[0]["train_loss"] and log["best_hold_out_loss"] < e[0]["hold_out_loss"])}
    print(f"G1 · seed {seed}: training loss {e[0]['train_loss']:.4f} → {e[-1]['train_loss']:.4f}; hold-out loss {e[0]['hold_out_loss']:.4f} → "
          f"{log['best_hold_out_loss']:.4f} at epoch {log['best_epoch'] + 1} (accuracy {G1[seed]['hold_out_accuracy_at_best']:.4f}): {G1[seed]['ok']}", flush=True)
    if not G1[seed]["ok"] and not TEST:
        raise SystemExit(f"stop: G1 failed for seed {seed}")


for _i, _seed in enumerate(SV["seeds"]):
    ensure_sv(_seed)
    if _i == 0:
        PREVIEW = table([_seed], DECISION_LEVEL, SYSTEMS)
        GV0 = g_view(PREVIEW)
        show(PREVIEW, SYSTEMS, f"PREVIEW with seed {_seed} alone (Split A at −6 dB; not a decision, the other seeds are trained whatever it shows):")
        print("G-VIEW after the first seed · the view alone reaches the DCASE 2020 baseline for every type (by the paper's score or by LDN1):",
              all(v["ok"] for v in GV0.values()), GV0, flush=True)
        if not TEST and not all(v["ok"] for v in GV0.values()):
            raise SystemExit("stop: G-VIEW failed after the first seed. Something is broken in the view; nothing is decided")
AUDIO = LABELS = None
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print(f"the trainings are done · {(time.time() - T_START) / 60:.0f} min in this session")

In [ ]:
# 8 · The decision at −6 dB (the rules: the program's keep rule on Split A; form ZP as notebook 25 selected it, no new selection)
SEEDS, DL = list(SV["seeds"]), DECISION_LEVEL
CLIP_SC, RES, CAL, PER_SEED, CARRIED = {}, {}, {}, {}, {}             # (level, type, machine) → every test clip's scores, saved in the zip
RES[DL] = table(SEEDS, DL, SYSTEMS, keep=CLIP_SC)
CAL[DL], CARRIED[DL] = spreads(SEEDS, DL), carried_over(SEEDS, DL)
_co = lambda d: {t: (round(d[t]["assigned_to_own_machine"], 3), round(d[t]["median_paper_score"], 3)) for t in TYPES}
print("the held-out normal clips at −6 dB (share the view assigns to their own machine, median paper's score):", _co(CARRIED[DL]))
_cs = lambda d: {t: (d[t]["n"], None if d[t]["sigma"] is None else round(d[t]["sigma"], 4)) for t in TYPES}
print("spreads of the log score on normal training clips at −6 dB (clips, σ): frozen, every training clip against its machine's others",
      _cs(CAL[DL]["frozen"]), "· view, the paper's score on the held-out clips", _cs(CAL[DL]["view_paper"]), "· view, B1 on the held-out clips (for ZB)",
      _cs(CAL[DL]["view_B1"]))
print("  for comparison, notebook 25 on DCASE's lists (σ): frozen, on clips its models never saw", N25_SIGMA["frozen"], "· view", N25_SIGMA["view_paper"])
GV = g_view(RES[DL])
print(f"G-VIEW · the view alone, {len(SEEDS)} seeds, reaches the DCASE 2020 baseline for every type (by the paper's score or by LDN1):",
      all(v["ok"] for v in GV.values()), GV)
if not TEST and not all(v["ok"] for v in GV.values()):
    raise SystemExit("stop: G-VIEW failed. Something is broken in the view; nothing is decided")
SC = {lv: {} for lv in LVL}
SC[DL] = {nm: s2(RES[DL][nm]) for nm in RES[DL]}
CUR = SC[DL]["frozen"]                                               # this session's recomputation (gate G0(c) tied it to notebook 23's)
THR = KEEP_AT if not TEST else (round(CUR[0] + BAR, 4), round(CUR[1] - BAR, 4))   # the rules' two bars (the synthetic test derives its own)
if not TEST and CUR != CURRENT_A:
    print(f"NOTE: the frozen system's scores recomputed here are {CUR}; the rules quote {CURRENT_A}. The rules' bars {KEEP_AT} are used.")
KEPT = bool(SC[DL]["ZP"][0] >= THR[0] and SC[DL]["ZP"][1] >= THR[1])
FORCED = False
if TEST and FORCE in ("keep", "drop"):
    KEPT, FORCED = FORCE == "keep", True
show(RES[DL], SYSTEMS, f"\nSplit A at −6 dB · AUC per type (mean over a type's four machines), the view with {len(SEEDS)} seeds:")
print("  the view alone in notebook 25 (DCASE's split of the same recordings, the paper's score):", N25_VIEW_DEV)
PER_SEED[DL] = {s: table([s], DL, ["view_paper", "ZP"]) for s in SEEDS}
print("\nSeed spread · ZP with each seed alone (development, confirmation):", {s: s2(PER_SEED[DL][s]["ZP"]) for s in SEEDS},
      "· the view alone (the paper's score):", {s: s2(PER_SEED[DL][s]["view_paper"]) for s in SEEDS})
DECISION = {"level": DL, "split": "A", "current": list(CUR), "keep_at": list(THR), "form": "ZP", "scores": list(SC[DL]["ZP"]), "kept": KEPT,
            "passes_development": bool(SC[DL]["ZP"][0] >= THR[0]), "passes_confirmation": bool(SC[DL]["ZP"][1] >= THR[1]), "test_forced": FORCED}
print(f"\nDECISION at −6 dB · frozen + view (form ZP): development {SC[DL]['ZP'][0]:.4f} against {THR[0]:.4f} needed (now {CUR[0]:.4f}); "
      f"confirmation {SC[DL]['ZP'][1]:.4f} against {THR[1]:.4f} needed (now {CUR[1]:.4f}) → "
      + ("KEPT" if KEPT else "NOT KEPT") + (" (forced by the test)" if FORCED else ""))

In [ ]:
# 9 · The per-level table (reading 3): only if the keep rule passed. The kept system is read once at 6 and 0 dB.
def cells_of(lv):
    # a level's four cells: the fused system's Split-A value against arXiv 2309.07500, beside reading 1 (notebook 23)
    return {t: {"A": RES[lv]["ZP"][t]["A"]["avg"], "published": PUBLISHED[lv][t], "reached": bool(RES[lv]["ZP"][t]["A"]["avg"] >= PUBLISHED[lv][t]),
                "reading_1": N23_LEVELS[lv][t]["A"]["avg"], "reading_1_reached": bool(N23_LEVELS[lv][t]["A"]["avg"] >= PUBLISHED[lv][t])} for t in TYPES}


READING3 = {"read": False, "levels": {}, "reason": None}
if not KEPT:
    STOP_PREFETCH.set()
    READING3["reason"] = "the fused system was not kept at −6 dB, so 6 and 0 dB are not read"
    print("Reading 3 is not made:", READING3["reason"])
else:
    for lv in OTHER_LEVELS:
        if not all(feats_ok(s, lv) for s in SEEDS):
            level_audio(lv)
        for s in SEEDS:
            ensure_feats(s, lv)
        RES[lv] = table(SEEDS, lv, SYSTEMS + ["ZP_m6cal"], keep=CLIP_SC)
        CAL[lv], CARRIED[lv] = spreads(SEEDS, lv), carried_over(SEEDS, lv)
        SC[lv] = {nm: s2(RES[lv][nm]) for nm in RES[lv]}
        PER_SEED[lv] = {s: table([s], lv, ["view_paper", "ZP"]) for s in SEEDS}
        show(RES[lv], SYSTEMS + ["ZP_m6cal"], f"\nSplit A at {LV_NAME[lv]} · AUC per type:")
        print("  spreads at this level (clips, σ): frozen", _cs(CAL[lv]["frozen"]), "· view, the paper's score", _cs(CAL[lv]["view_paper"]),
              "· view, B1", _cs(CAL[lv]["view_B1"]))
        print(f"  the held-out normal clips at {LV_NAME[lv]} (share the view assigns to their own machine, median paper's score):", _co(CARRIED[lv]),
              "· at −6 dB:", _co(CARRIED[DL]))
        for t in TYPES:                                               # this level's audio is not needed again in this session
            ST[t].pop(lv, None)
        gc.collect()
    READING3["levels"] = {lv: cells_of(lv) for lv in LVL}
    _reached = {(lv, t): READING3["levels"][lv][t]["reached"] for lv in LVL for t in TYPES}
    _others = [k for k in _reached if k not in TARGETS]
    n_t, lost = sum(_reached[k] for k in TARGETS), [k for k in _others if READING3["levels"][k[0]][k[1]]["reading_1_reached"] and not _reached[k]]
    READING3.update({"read": True, "cells_reached": int(sum(_reached.values())), "targets": {f"{NAME[t]} at {LV_NAME[lv]}": bool(_reached[(lv, t)]) for lv, t in TARGETS},
                     "cells_lost": [f"{NAME[t]} at {LV_NAME[lv]}" for lv, t in lost],
                     "outcome": "answered" if n_t == len(TARGETS) and not lost else ("answered in part" if n_t >= 1 else "not answered"),
                     "carried_spreads": {lv: {t: {"A": RES[lv]["ZP_m6cal"][t]["A"]["avg"], "reached": bool(RES[lv]["ZP_m6cal"][t]["A"]["avg"] >= PUBLISHED[lv][t])}
                                              for t in TYPES} for lv in OTHER_LEVELS},
                     "reported_ZB": {lv: {t: {"A": RES[lv]["ZB"][t]["A"]["avg"], "reached": bool(RES[lv]["ZB"][t]["A"]["avg"] >= PUBLISHED[lv][t])}
                                          for t in TYPES} for lv in LVL if "ZB" in RES[lv]}})
    print("\nREADING 3 · the kept system (frozen + view, form ZP) on Split A, against arXiv 2309.07500 (✗ = below the published number; reading 1 in brackets):")
    for lv in LVL:
        print(f"  {LV_NAME[lv]:>6}: " + " · ".join(f"{NAME[t]} {c['A']:.4f}{'' if c['reached'] else '✗'} / {c['published']:.4f} ({c['reading_1']:.4f})"
                                                    for t, c in READING3["levels"][lv].items()))
    print(f"  cells reached: {READING3['cells_reached']} of 12 · the two cells reading 1 missed: {READING3['targets']} · cells lost: {READING3['cells_lost'] or 'none'}"
          f" → weak point 2: {READING3['outcome'].upper()}")
    print("  with the −6 dB spreads carried to 6 and 0 dB instead (reported; the result is the table above):",
          {lv: {NAME[t]: v["A"] for t, v in READING3["carried_spreads"][lv].items()} for lv in OTHER_LEVELS})
    print("  ZB, the view by B1 against the level's references instead of the paper's score (reported; it decides and answers nothing):",
          {lv: {NAME[t]: f"{v['A']:.4f}{'' if v['reached'] else '✗'}" for t, v in d.items()} for lv, d in READING3["reported_ZB"].items()})

In [ ]:
# 10 · Looked after (no decision), save, one picture, and one zip to download
import matplotlib.pyplot as plt


def looked_after(seeds, lv=DECISION_LEVEL):
    # for each machine at −6 dB, in the view's space: the reference clip that is nearest to the most abnormal test clips, and the three
    # normal test clips with the highest paper's score (the question notebook 25's record left open, asked of this view)
    out = {}
    for t in TYPES:
        V, names, label, held = view_matrix(seeds, lv, t), META[t][0], META[t][2], set(int(r) for r in HOLD_ROWS[t]); out[t] = {}
        for p in IDS[t]:
            tr, te = SPLIT[t][p]; U = unit(V[tr]); y = label[te]
            near = (unit(V[te]) @ U.T).argmax(1)
            Srr = U @ U.T; np.fill_diagonal(Srr, -np.inf); d1 = 1.0 - Srr.max(1)
            cnt = np.bincount(near[y == 1], minlength=len(tr)); h = int(cnt.argmax())
            ps = paper(seeds, lv, t, te); nor = np.where(y == 0)[0]; top = nor[np.argsort(-ps[nor])[:3]]
            out[t][p] = {"reference_nearest_to_most_abnormal_clips": {
                             "file": str(names[tr[h]]), "abnormal_test_clips_nearest_to_it": int(cnt[h]), "abnormal_test_clips": int(np.sum(y == 1)),
                             "normal_test_clips_nearest_to_it": int(np.sum(near[y == 0] == h)), "its_own_nearest_neighbour_distance": float(d1[h]),
                             "median_of_that_distance_over_the_references": float(np.median(d1)), "held_out": bool(int(tr[h]) in held),
                             "its_paper_score": float(paper(seeds, lv, t, tr[[h]])[0])},
                         "normal_test_clips_with_the_highest_paper_score": [{"file": str(names[te[i]]), "paper_score": float(ps[i])} for i in top],
                         "median_paper_score_of_the_normal_test_clips": float(np.median(ps[nor]))}
        VM.clear()
    return out


LOOKED = looked_after(SEEDS)
_worst = max(((v["reference_nearest_to_most_abnormal_clips"]["abnormal_test_clips_nearest_to_it"] / v["reference_nearest_to_most_abnormal_clips"]["abnormal_test_clips"], t, p)
              for t in TYPES for p, v in LOOKED[t].items()))
print(f"looked after · the reference clip that draws the largest share of a machine's abnormal test clips: {NAME[_worst[1]]} {_worst[2]}, "
      f"{_worst[0]:.0%} →", LOOKED[_worst[1]][_worst[2]]["reference_nearest_to_most_abnormal_clips"])

GATE = json.loads(GATE_F.read_text()) if GATE_F.exists() else {}
G0["b"] = {"ok": bool(all(GATE.get(f"{t}|{lv}", {}).get("names_are_notebook_20s") for t in TYPES for lv in ([DL] + (OTHER_LEVELS if READING3["read"] else [])))),
           "this_session": G0B, "recorded_when_the_stores_were_built": GATE}
print("G0(b) · every zip had Zenodo's MD5 and notebook 20's clip names, at every level that was read:", G0["b"]["ok"])
READ_LEVELS = [lv for lv in LVL if lv in RES]
RESULT = {"rules": RULES, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST,
          "meta": {"device": DEVICE, "gpu": GPU, "gpu_gb": round(GPU_GB, 1), "torch": torch.__version__, "torchaudio": torchaudio.__version__,
                   "reference": {"repository": "https://github.com/soonhyeon/Noisy-ArcMix", "commit": REF_COMMIT, "licence": "MIT, Copyright (c) 2023 soonhyeon"},
                   "sv": SV, "n_frames": int(N_FRAMES), "last_kernel": list(LAST_KERNEL), "classes": CLASSES, "n_clips": int(len(ITEMS)),
                   "zips": {k: list(v[1:]) for k, v in ZIPS.items()}, "additional_zips": {k: list(v[1:]) for k, v in XZIPS.items()},
                   "memory_layout_seconds_per_step": MF_TIMES, "prep_seconds": PREP_S, "train_seconds": T_TRAIN, "embed_seconds": T_EMBED,
                   "session_minutes": round((time.time() - T_START) / 60, 1), "drive_free_gb": round(DRIVE_FREE_GB, 1)},
          "G0": G0, "G1": G1, "G_VIEW": {"first_seed": GV0, "three_seeds": GV},
          "levels_read": READ_LEVELS, "systems": {lv: RES[lv] for lv in READ_LEVELS}, "scores": {lv: {nm: list(v) for nm, v in SC[lv].items()} for lv in READ_LEVELS},
          "calibration": {lv: CAL[lv] for lv in READ_LEVELS}, "minimum_clips": CAL_MIN, "held_out_clips_at_each_level": {lv: CARRIED[lv] for lv in READ_LEVELS}, "per_seed": {lv: PER_SEED[lv] for lv in READ_LEVELS},
          "preview_first_seed": PREVIEW, "decision": DECISION, "reading_3": READING3, "looked_after": LOOKED,
          "frozen_all_splits": FROZEN_RES, "notebook_23": {"levels": N23_LEVELS, "current": list(CURRENT_A), "keep_at": list(KEEP_AT)},
          "published": PUBLISHED, "targets": [f"{NAME[t]} at {LV_NAME[lv]}" for lv, t in TARGETS],
          "notebook_25": {"sigma": N25_SIGMA, "view_alone_development": N25_VIEW_DEV}, "dcase_baseline_development": DCASE_BASELINE_DEV,
          "training_logs": LOGS}
(OUT / "second_view_mimii.json").write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))
# every test clip's score under every system at every level that was read, so that each number above can be recomputed from the zip
SC_DIR = OUT / "test_scores"; SC_DIR.mkdir(exist_ok=True)
for f in SC_DIR.glob("scores_*.csv"):
    f.unlink()
for (lv, t, p), d in CLIP_SC.items():
    cols = [nm for nm in SYSTEMS + ["ZP_m6cal"] if nm in d]
    (SC_DIR / f"scores_{LEVEL_CODE[lv]}_{t}_{p}.csv").write_text("file,label," + ",".join(cols) + "\n" + "".join(
        f"{d['file'][i]},{int(d['label'][i])}," + ",".join(repr(float(d[nm][i])) for nm in cols) + "\n" for i in range(len(d["file"]))))
# the clips the spreads rest on, at every level that was read: the frozen system's left-out score of every normal training clip, and the
# held-out clips under the view (the paper's score, how many seeds assign the clip to its own machine, its left-out B1). No test clip.
CAL_DIR = OUT / "calibration_scores"; CAL_DIR.mkdir(exist_ok=True)
for f in CAL_DIR.glob("*.csv"):
    f.unlink()
for lv in READ_LEVELS:
    for t in TYPES:
        _nm = META[t][0]
        (CAL_DIR / f"frozen_{LEVEL_CODE[lv]}_{t}.csv").write_text("file,frozen_left_out\n" + "".join(
            f"{_nm[r]},{float(v)!r}\n" for p in IDS[t] for r, v in zip(TRAIN_ROWS[(t, p)], LOO_F[(lv, t, p)])))
        _rows = np.asarray(HOLD_ROWS[t], dtype=np.int64); _pa = paper(SEEDS, lv, t, _rows); _ho = b1_held_out(SEEDS, lv, t)
        _own = sum(HITS[(s, lv)][t][_rows].astype(np.int64) for s in SEEDS)
        (CAL_DIR / f"held_out_{LEVEL_CODE[lv]}_{t}.csv").write_text("file,paper,seeds_assigning_own_machine,view_B1_left_out\n" + "".join(
            f"{_nm[r]},{float(a)!r},{int(o)},{repr(float(_ho[int(r)])) if int(r) in _ho else ''}\n" for r, a, o in zip(_rows, _pa, _own)))

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
COL = {"frozen": "#2a78d6", "view": "#eb6834", "fused": "#1baf7a"}    # three categorical slots, checked with the palette validator
MARK = {"frozen": "o", "view": "s", "fused": "D"}                    # a shape per system as well as a colour


def dots(ax, xs, vals, k, lab):
    # every mark carries its value, on a white patch so that a dashed line behind it does not cross the digits
    ax.plot(xs, vals, MARK[k], ms=9, color=COL[k], mec="white", mew=1.2, label=lab, zorder=3)
    span = ax.get_ylim()[1] - ax.get_ylim()[0]
    for a, v in zip(xs, vals):
        ax.text(a, v + 0.035 * span, f"{v:.4f}", ha="center", va="bottom", fontsize=8, color=INK, zorder=4,
                bbox=dict(facecolor="white", edgecolor="none", pad=0.6, alpha=0.9))


fig, axes = plt.subplots(3, 1, figsize=(10.5, 12.8))                  # one wide panel per level: three close values stay apart
x, w = np.arange(len(TYPES)), 0.27
for ax, lv in zip(axes, LVL):
    pub, old = [PUBLISHED[lv][t] for t in TYPES], [N23_LEVELS[lv][t]["A"]["avg"] for t in TYPES]
    read = lv in RES
    vals = [v for v in pub + old] + ([RES[lv][nm][t]["A"]["avg"] for nm in ("view_paper", "ZP") for t in TYPES] if read else [])
    lo = math.floor((min(vals) - 0.1 * (1.0 - min(vals)) - 0.004) * 200) / 200
    ax.set_ylim(lo, 1.0 + 0.17 * (1.0 - lo))
    if read:
        dots(ax, x - w, old, "frozen", "frozen system (notebook 23, reading 1)")
        dots(ax, x, [RES[lv]["view_paper"][t]["A"]["avg"] for t in TYPES], "view", "second view alone (the paper's score)")
        dots(ax, x + w, [RES[lv]["ZP"][t]["A"]["avg"] for t in TYPES], "fused", "frozen + view, form ZP" + (" (kept; reading 3)" if KEPT else " (not kept)"))
        n_ok = sum(RES[lv]["ZP"][t]["A"]["avg"] >= PUBLISHED[lv][t] for t in TYPES)
        ttl = (f"{LV_NAME[lv]} · Split A · frozen + view reaches {n_ok} of the 4 published numbers "
               f"(reading 1: {sum(o >= p_ for o, p_ in zip(old, pub))} of 4)")
        if lv == DL:
            ttl += (f"\nthe keep rule: pumps and fans {CUR[0]:.4f} → {SC[DL]['ZP'][0]:.4f} (needs {THR[0]:.4f}); "
                    f"valves and slide rails {CUR[1]:.4f} → {SC[DL]['ZP'][1]:.4f} (needs {THR[1]:.4f})")
    else:
        dots(ax, x, old, "frozen", "frozen system (notebook 23, reading 1)")
        ttl = (f"{LV_NAME[lv]} · Split A · not read again: the fused system was not kept at −6 dB "
               f"(reading 1: {sum(o >= p_ for o, p_ in zip(old, pub))} of 4)")
    ax.hlines(pub, x - 0.44, x + 0.44, color=INK, lw=1.2, ls=(0, (4, 2)), label="arXiv 2309.07500, one microphone (another split; indicative)", zorder=2)
    ax.set_title(ttl, loc="left", color=INK, fontsize=10)
    ax.set_xticks(x, [NAME[t] for t in TYPES]); ax.set_xlim(-0.6, len(TYPES) - 0.4)
    ax.set_ylabel("AUC (the scale does not start at zero)", color=MUTED, fontsize=9)
    ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True); ax.tick_params(colors=MUTED)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
_h, _l = axes[-1].get_legend_handles_labels()                         # the −6 dB panel always holds every series
fig.legend(_h, _l, frameon=False, fontsize=9.5, loc="lower center", ncol=2, bbox_to_anchor=(0.5, 0.004))
fig.suptitle("P04 · notebook 26 · the second view (TASTgram-MFN with Noisy-ArcMix, trained at −6 dB)\n"
             "fused with the single-microphone system · MIMII, microphone 0 · mean over a type's four machines", fontsize=11, x=0.012, y=0.992, ha="left")
fig.tight_layout(rect=(0, 0.05, 1, 0.968), h_pad=2.2); fig.savefig(OUT / "second_view_mimii_levels.png", dpi=150); plt.show()

ZIP = OUT / "p04_n26_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("second_view_mimii.json", "second_view_mimii_levels.png"):
        z.write(OUT / f, f)
    for d in (SC_DIR, CAL_DIR):
        for f in sorted(d.glob("*.csv")):
            z.write(f, f"{d.name}/{f.name}")
print(f"saved {ZIP} · {(time.time() - T_START) / 60:.0f} min in this session")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n26_results.zip`). It is also in Drive › `P04_SecondViewMIMII`.